
# ALICE Masterclass: Nuclear Modification Factor ($R_{AA}$ and $R_{CP}$)

**Pb–Pb collisions at $\sqrt{s_{NN}} = 2.76$ TeV (LHC10h), ALICE experiment**

This notebook walks step by step through a real ALICE masterclass analysis:

1. **Part A** — the original pipeline exactly as it runs today: event/centrality classification, charged-particle $p_T$ spectra, and $R_{AA}$ / $R_{CP}$ for charged tracks. It runs on the same default data as before, with three corrections: the statistical uncertainty of the $p_T$ spectra now includes the bin width (Section 2.4); the $p_T$ bin edges are the true edges of the pp reference (Section 2.4); and the pp uncertainties are the pp reference's own bin errors instead of a flat 10% (Section 2.5). Section 3.5 compares the result with ALICE's published $R_{AA}$.
2. **Part B** — the same $R_{AA}$/$R_{CP}$ mathematics repackaged into small, reusable, **particle-species-agnostic** functions, checked against Part A to prove they give identical numbers.
3. **Part C** — **the ALICE J/$\psi$ masterclass, step by step**: unlike charged hadrons, a J/$\psi$ isn't counted directly — it's reconstructed from its decay products.
4. 
### The physics: what is $R_{AA}$?

The nuclear modification factor compares particle production in Pb–Pb collisions to a pp reference, scaled by the number of underlying binary nucleon-nucleon collisions $\langle N_{coll} \rangle$:

$$
R_{AA}(p_T) = \frac{ \dfrac{1}{N_{evt}^{AA}} \dfrac{dN^{AA}}{dp_T} }
                    { \langle N_{coll} \rangle \, \dfrac{1}{N_{evt}^{pp}} \dfrac{dN^{pp}}{dp_T} }
$$

If Pb–Pb collisions were just an incoherent superposition of $N_{coll}$ independent pp collisions, $R_{AA} = 1$. Values below 1 (as seen for high-$p_T$ charged hadrons and for J/$\psi$) signal energy loss or suppression in the hot, dense medium (the quark–gluon plasma) created in the collision. Crucially, this formula does not care *what particle species* $N^{AA}$ and $N^{pp}$ count — that's the generalisation this notebook builds towards.

$R_{CP}$ replaces the pp reference with a peripheral (more glancing, less dense-medium) Pb–Pb centrality class, cancelling some common systematics:

$$
R_{CP}(p_T) = \frac{ \langle N_{coll} \rangle^{peripheral} / N_{evt}^{central} \; dN^{central}/dp_T }
                    { \langle N_{coll} \rangle^{central} / N_{evt}^{peripheral} \; dN^{peripheral}/dp_T }
$$



## Step 0 — Imports

Same imports as the original script.


In [ ]:
import ALICE_RAA_Tools as at
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
from matplotlib import colors as mcolors


## Step 0.1 — Configuration

The centrality binning and $\langle N_{coll}\rangle$ (Glauber-model) values below are unchanged from the original analysis, and are shared by **every** part of this notebook: $\langle N_{coll}\rangle$ describes the collision geometry, not the particle species being measured, so the exact same table is valid whether you're counting charged tracks or reconstructing J/$\psi$.


In [ ]:
# Available centralities:
# 0-5%, 0-10%, 5-10%, 10-20%, 20-30%, 30-40%, 40-50%, 50-60%, 60-70%, 70-80%
# Full 0-80% coverage with no gaps: 0-5 and 5-10 split out the lowest bin,
# then 10-20, 20-30, ..., 70-80 cover the rest contiguously. Events above 80% are not used.
listCentralities = [
    (0, 5), (5, 10),
    (10, 20), (20, 30), (30, 40), (40, 50),
    (50, 60), (60, 70), (70, 80)
]

# N_coll values per centrality class (from Glauber model)
dictNColl = {
    '0-5':   1686.87,
    '0-10':  1502.7,
    '5-10':  1319.89,
    '10-20':  923.89,
    '20-30':  558.68,
    '30-40':  321.20,
    '40-50':  171.67,
    '50-60':   85.13,
    '60-70':   38.51,
    '70-80':   15.78,
    'pp':       1.
}

## Step 0.2 — Saving the plots

Every figure this notebook shows is also saved to the folder `figures/`, as a PDF (for the thesis) and a PNG (for slides). The cell below wraps `plt.show()`, so no other cell needs changing. Files are numbered in the order the plots appear and named from the plot's title or axis labels, e.g. `03_R_AA_vs_p_T.pdf`. Re-running a plotting cell saves a new numbered copy; re-run this cell first to start the numbering again. Set `SAVE_PLOTS = False` to only display the plots.

In [ ]:
import os
import re

SAVE_PLOTS  = True            # False: only display the plots
FIG_DIR     = 'figures'       # folder the plots are saved in
FIG_FORMATS = ('pdf', 'png')  # pdf for the thesis, png for slides
FIG_DPI     = 200             # resolution of the png files

_original_show = getattr(plt.show, '_original', plt.show)   # the real plt.show, even if this cell is re-run
_fig_counter = 0

def _plain(text):
    # turn a matplotlib label into a file-name fragment: drop LaTeX commands and symbols
    text = text.split('=')[0]                       # 'R_{AA}=\frac{...}' -> 'R_{AA}'
    text = re.sub(r'\\mathrm\{([^}]*)\}', r'\1', text)   # '\mathrm{AA}' -> 'AA', so file names stay the same
    text = text.replace('\\', '')                    # '\psi' -> 'psi'
    text = re.sub(r'[^A-Za-z0-9]+', '_', text)
    return text.strip('_')

def _figure_name(fig):
    # a readable name: the figure or axes title, else the label in the plot, else 'y vs x'
    if fig._suptitle is not None and fig._suptitle.get_text().strip():
        return _plain(fig._suptitle.get_text())
    for ax in fig.axes:
        if ax.get_title().strip():
            return _plain(ax.get_title())
    for ax in fig.axes:
        if ax.texts:
            return _plain(max(ax.texts[0].get_text().splitlines(), key=len))
    ax = fig.axes[0] if fig.axes else None
    if ax is not None and ax.get_ylabel():
        return _plain(ax.get_ylabel()) + '_vs_' + _plain(ax.get_xlabel())
    return 'figure'

def save_open_figures():
    # save every figure that is open right now, then return the paths
    global _fig_counter
    paths = []
    for num in plt.get_fignums():
        fig = plt.figure(num)
        if not fig.axes:
            continue
        _fig_counter += 1
        stem = os.path.join(FIG_DIR, f'{_fig_counter:02d}_{_figure_name(fig)[:60]}')
        for ext in FIG_FORMATS:
            fig.savefig(f'{stem}.{ext}', dpi=FIG_DPI, bbox_inches='tight')
        paths.append(stem)
    return paths

def _show_and_save(*args, **kwargs):
    if SAVE_PLOTS:
        for stem in save_open_figures():
            print(f'saved {stem}.{{{",".join(FIG_FORMATS)}}}')
    return _original_show(*args, **kwargs)

_show_and_save._original = _original_show
plt.show = _show_and_save
os.makedirs(FIG_DIR, exist_ok=True)
print(f"Plots will be saved to '{FIG_DIR}/' as {', '.join(FIG_FORMATS)}." if SAVE_PLOTS else 'Plots are only displayed, not saved.')


### Configurable data inputs

The original script hard-coded three file names inline. They're pulled out here as plain variables with the **same default values**, so Part A below behaves identically to the original script by default — but you can now point these at your own equivalent files (e.g. a different run, or Monte Carlo) without touching any analysis logic.


In [ ]:
# Point these at your own files if you have them; the defaults match the
# files the original script used, so nothing below changes unless you do.
EVENT_FILE = 'event_information.csv'
TRACK_FILE = 'track_info.pkl'
PP_REF_FILE = 'pp_reference.dat'
PP_REF_ROOT_FILE = 'PP_2760GeV_BaseLine.root'   # the ALICE masterclass pp reference histogram that pp_reference.dat
                                                # was taken from; it supplies the pp bin edges and uncertainties


---
# Part A — Charged-Hadron $R_{AA}$ (the original pipeline)

Everything in Part A reproduces the original script's logic and results exactly, cell by cell, with explanations added. If you have the input files (`event_information.csv`/`EVENT_FILE`, `track_info.pkl`/`TRACK_FILE`, `pp_reference.dat`/`PP_REF_FILE`, `PP_2760GeV_BaseLine.root`/`PP_REF_ROOT_FILE`, and the `ALICE_RAA_Tools` module) in the working directory, this part runs standalone.



## 1. Event Analysis

### 1.1 — Build the centrality-class dictionary

`at.create_dictionary` turns the list of `(low, high)` centrality edges above into the lookup structure the rest of the analysis uses to classify events.


In [ ]:
dictCentralities = at.create_dictionary(listCentralities)


### 1.2 — Load events

Each row of the event file is one collision, with its charged-track multiplicity (`eventMult`) and centrality percentile (`eventCent`).


In [ ]:
event_lines = at.count_lines(EVENT_FILE)
print(f"Number of events: '{event_lines}'!")

df_events = pd.read_csv(EVENT_FILE, header=None, names=['eventMult', 'eventCent'])
print(df_events.head())


### 1.3 — Tag each event with its centrality class

A boolean column is added per centrality bin, then `at.find_centralities` determines which bin(s) each event's `eventCent` falls into. `dictEventsCent` accumulates the number of events per centrality class — this event count is what the $p_T$ spectra get normalised by in Section 2.


In [ ]:
for key in dictCentralities:
    df_events[key] = False

print(df_events.head())

dictEventsCent = {}
for row in df_events.iterrows():
    keys = at.find_centralities(dictCentralities, row[1]['eventCent'])
    if keys:
        for cent in keys:
            df_events.loc[row[0], cent] = True
            if cent in dictEventsCent:
                dictEventsCent[cent] += 1
            else:
                dictEventsCent[cent] = 1

print(df_events.head())
print(dictEventsCent)


### 1.4 — Multiplicity histograms per centrality

One TPC-track-multiplicity histogram per centrality class, plus the inclusive 0–100% distribution, with Poisson ($\sqrt{N}$) statistical errors.


In [ ]:
hTPC_0_100, bins = at.create_event_histos(df_events['eventMult'])
hTPC_0_5,   bins = at.create_event_histos(df_events[df_events['0-5']   == True]['eventMult'])
hTPC_5_10,  bins = at.create_event_histos(df_events[df_events['5-10']  == True]['eventMult'])
hTPC_10_20, bins = at.create_event_histos(df_events[df_events['10-20'] == True]['eventMult'])
hTPC_20_30, bins = at.create_event_histos(df_events[df_events['20-30'] == True]['eventMult'])
hTPC_30_40, bins = at.create_event_histos(df_events[df_events['30-40'] == True]['eventMult'])
hTPC_40_50, bins = at.create_event_histos(df_events[df_events['40-50'] == True]['eventMult'])
hTPC_50_60, bins = at.create_event_histos(df_events[df_events['50-60'] == True]['eventMult'])
hTPC_60_70, bins = at.create_event_histos(df_events[df_events['60-70'] == True]['eventMult'])
hTPC_70_80, bins = at.create_event_histos(df_events[df_events['70-80'] == True]['eventMult'])

hTPC_0_100_err = np.sqrt(hTPC_0_100)
hTPC_0_5_err   = np.sqrt(hTPC_0_5)
hTPC_5_10_err  = np.sqrt(hTPC_5_10)
hTPC_10_20_err = np.sqrt(hTPC_10_20)
hTPC_20_30_err = np.sqrt(hTPC_20_30)
hTPC_30_40_err = np.sqrt(hTPC_30_40)
hTPC_40_50_err = np.sqrt(hTPC_40_50)
hTPC_50_60_err = np.sqrt(hTPC_50_60)
hTPC_60_70_err = np.sqrt(hTPC_60_70)
hTPC_70_80_err = np.sqrt(hTPC_70_80)

### 1.5 — Plot multiplicity distributions

In [ ]:
plt.figure()
plt.errorbar(bins[:-1], hTPC_0_100, yerr=hTPC_0_100_err, marker='+', c='black',      label='0-100%')
plt.errorbar(bins[:-1], hTPC_0_5,   yerr=hTPC_0_5_err,   marker='+', c='darkred',    label='0-5%')
plt.errorbar(bins[:-1], hTPC_5_10,  yerr=hTPC_5_10_err,  marker='+', c='red',        label='5-10%')
plt.errorbar(bins[:-1], hTPC_10_20, yerr=hTPC_10_20_err, marker='+', c='darkorange', label='10-20%')
plt.errorbar(bins[:-1], hTPC_20_30, yerr=hTPC_20_30_err, marker='+', c='gold',       label='20-30%')
plt.errorbar(bins[:-1], hTPC_30_40, yerr=hTPC_30_40_err, marker='+', c='yellowgreen',label='30-40%')
plt.errorbar(bins[:-1], hTPC_40_50, yerr=hTPC_40_50_err, marker='+', c='green',      label='40-50%')
plt.errorbar(bins[:-1], hTPC_50_60, yerr=hTPC_50_60_err, marker='+', c='deepskyblue',label='50-60%')
plt.errorbar(bins[:-1], hTPC_60_70, yerr=hTPC_60_70_err, marker='+', c='blue',       label='60-70%')
plt.errorbar(bins[:-1], hTPC_70_80, yerr=hTPC_70_80_err, marker='+', c='navy',       label='70-80%')
plt.xlabel('Number of TPC tracks', fontsize=14)
plt.ylabel('Counts')
plt.ylim(1, 2E4)
plt.yscale('log')
plt.legend(loc='upper right')
plt.savefig('TPC_tracks.pdf')
plt.show()


### 1.6 — 2D check: multiplicity vs. centrality

A quick sanity-check plot: multiplicity should fall monotonically as the centrality percentile increases (more peripheral = fewer tracks).

ALICE calibrates centrality only down to 90%. The few events at exactly 100% have no valid centrality and are not used: every class used here lies between 0% and 80%.

In [ ]:
plt.figure()
plt.hist2d(
    df_events['eventCent'], df_events['eventMult'],
    bins=(100, 200), range=[[0, 100], [0, 2000]],
    cmap=plt.cm.jet, norm=mpl.colors.LogNorm()
)
cbar = plt.colorbar()
cbar.set_label('Number of events per bin')
plt.xlabel('Centrality (%)', fontsize=14)
plt.ylabel('Number of TPC tracks')
plt.show()


## 2. Track Momentum Spectra

### 2.1–2.3 — Load tracks and sort by centrality

This is the computationally heavy step: every reconstructed track (tens of millions of them) is assigned to the centrality class(es) of the event it came from.


In [ ]:
print("Number of tracks: '38042122'! (~600 MB)")

dictMomCent = {}
for key in dictCentralities:
    dictMomCent[key] = []

df_tracks = pd.read_pickle(TRACK_FILE, 'bz2').to_numpy()

for line in df_tracks:
    keys = at.find_centralities(dictCentralities, line[1])
    if keys:
        for key in keys:
            dictMomCent[key].append(line[0])


### 2.4 — Build $p_T$ histograms, normalised per event and bin width

`make_spectrum` turns a raw list of track momenta into a proper differential yield: histogram counts divided by both the bin width and the number of events in that centrality class, i.e. $\frac{1}{N_{evt}}\frac{dN}{dp_T}$.

The statistical uncertainty is the Poisson uncertainty of the count, $\sigma_N = \sqrt{N}$, normalised the same way as the yield:

$$
Y = \frac{N}{N_{evt}\,\Delta p_T}, \qquad \sigma_Y = \frac{\sqrt{N}}{N_{evt}\,\Delta p_T}, \qquad \frac{\sigma_Y}{Y} = \frac{1}{\sqrt{N}} .
$$

**Bin edges.** The 51 bins run from 0.15 to 15 GeV/$c$ (widths 0.05 GeV/$c$ up to 1 GeV/$c$, then 0.1, 0.2, 0.5 and 1 GeV/$c$) — exactly the bins of the pp reference histogram, so each Pb–Pb bin is divided by the pp value of the same interval. Some copies of `ALICE_RAA_Tools.get_bins()` return the bin centres (0.175, 0.225, …, 15.5) instead of the edges; that shifts every Pb–Pb bin half a bin upwards against the pp reference and pulls $R_{AA}$ down by 10–50% above 1 GeV/$c$, so the edges are set explicitly here. Points are drawn at the bin centres.


In [ ]:
# pT bin edges (GeV/c): the binning of the pp reference (PP_2760GeV_BaseLine.root, histogram trackPt_pp),
# which is also ALICE's published binning. at.get_bins() in some copies of ALICE_RAA_Tools.py returns the
# bin CENTRES (0.175, 0.225, ...) instead, which shifts every Pb-Pb bin by half a bin against the pp reference.
binsPt      = [0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95,
          1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6,
          3.8, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]
binsPtCenter = 0.5 * (np.asarray(binsPt[:-1]) + np.asarray(binsPt[1:]))   # where the points are drawn
binWidth    = at.get_bin_width(binsPt)
x_bin_width = np.asarray(binWidth) / 2

def make_spectrum(mom_list, n_events):
    """Return (yield, stat_error) normalised per event per bin width."""
    h, _ = np.histogram(mom_list, binsPt)
    err  = np.sqrt(h) / np.asarray(binWidth) / n_events   # Poisson sqrt(N), normalised exactly like the yield
    h    = h / np.asarray(binWidth) / n_events
    return h, err

hPbPb_0_5,   hPbPb_0_5_err   = make_spectrum(dictMomCent['0-5'],   dictEventsCent['0-5'])
hPbPb_5_10,  hPbPb_5_10_err  = make_spectrum(dictMomCent['5-10'],  dictEventsCent['5-10'])
hPbPb_10_20, hPbPb_10_20_err = make_spectrum(dictMomCent['10-20'], dictEventsCent['10-20'])
hPbPb_20_30, hPbPb_20_30_err = make_spectrum(dictMomCent['20-30'], dictEventsCent['20-30'])
hPbPb_30_40, hPbPb_30_40_err = make_spectrum(dictMomCent['30-40'], dictEventsCent['30-40'])
hPbPb_40_50, hPbPb_40_50_err = make_spectrum(dictMomCent['40-50'], dictEventsCent['40-50'])
hPbPb_50_60, hPbPb_50_60_err = make_spectrum(dictMomCent['50-60'], dictEventsCent['50-60'])
hPbPb_60_70, hPbPb_60_70_err = make_spectrum(dictMomCent['60-70'], dictEventsCent['60-70'])
hPbPb_70_80, hPbPb_70_80_err = make_spectrum(dictMomCent['70-80'], dictEventsCent['70-80'])


### 2.5 — Load the pp reference spectrum

`pp_reference.dat` holds the 2.76 TeV pp reference yield, $\frac{1}{N_{evt}}\frac{dN}{dp_T}$, in the same 51 bins. Its values are identical to the histogram `trackPt_pp` in `PP_2760GeV_BaseLine.root`, the pp reference of the ALICE $R_{AA}$ masterclass. The uncertainties are taken from that histogram (between 5.6% and 10.4% per bin, median 7.6%) instead of the flat 10% the original script assumed; the cell checks that the histogram's bins and values match before using them. If the ROOT file is not available, the cell falls back to the flat 10%.


In [ ]:
import os
pp = np.genfromtxt(PP_REF_FILE)

if os.path.exists(PP_REF_ROOT_FILE):
    try:
        import uproot  # reads ROOT files in pure Python
    except ImportError:
        import subprocess, sys, importlib
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'uproot'])
        importlib.invalidate_caches()
        import uproot
    h_pp = uproot.open(PP_REF_ROOT_FILE)['trackPt_pp']
    vals, edges = h_pp.to_numpy()
    errs = h_pp.errors()
    sel = (edges[:-1] >= binsPt[0] - 1e-9) & (edges[1:] <= binsPt[-1] + 1e-9)
    assert np.allclose(edges[:-1][sel], binsPt[:-1]) and np.allclose(edges[1:][sel], binsPt[1:]), 'pp bins differ from binsPt'
    assert np.allclose(vals[sel], pp, rtol=1e-4), 'pp_reference.dat differs from the histogram'
    pp_err = errs[sel]
    print(f'pp uncertainties from {PP_REF_ROOT_FILE}: {100*np.min(pp_err/pp):.1f}% to {100*np.max(pp_err/pp):.1f}% per bin')
else:
    pp_err = np.divide(pp, 10)
    print(f'{PP_REF_ROOT_FILE} not found: using a flat 10% uncertainty on the pp reference')
print(pp)
print(pp)

### 2.6 — Plot $p_T$ spectra for all centralities and pp

In [ ]:
plt.figure()
plt.errorbar(binsPtCenter, hPbPb_0_5,   xerr=x_bin_width, yerr=hPbPb_0_5_err,   marker='+', c='darkred',    label='0-5%')
plt.errorbar(binsPtCenter, hPbPb_5_10,  xerr=x_bin_width, yerr=hPbPb_5_10_err,  marker='+', c='red',        label='5-10%')
plt.errorbar(binsPtCenter, hPbPb_10_20, xerr=x_bin_width, yerr=hPbPb_10_20_err, marker='+', c='darkorange', label='10-20%')
plt.errorbar(binsPtCenter, hPbPb_20_30, xerr=x_bin_width, yerr=hPbPb_20_30_err, marker='+', c='gold',       label='20-30%')
plt.errorbar(binsPtCenter, hPbPb_30_40, xerr=x_bin_width, yerr=hPbPb_30_40_err, marker='+', c='yellowgreen',label='30-40%')
plt.errorbar(binsPtCenter, hPbPb_40_50, xerr=x_bin_width, yerr=hPbPb_40_50_err, marker='+', c='green',      label='40-50%')
plt.errorbar(binsPtCenter, hPbPb_50_60, xerr=x_bin_width, yerr=hPbPb_50_60_err, marker='+', c='deepskyblue',label='50-60%')
plt.errorbar(binsPtCenter, hPbPb_60_70, xerr=x_bin_width, yerr=hPbPb_60_70_err, marker='+', c='blue',       label='60-70%')
plt.errorbar(binsPtCenter, hPbPb_70_80, xerr=x_bin_width, yerr=hPbPb_70_80_err, marker='+', c='navy',       label='70-80%')
plt.errorbar(binsPtCenter, pp,           xerr=x_bin_width, yerr=pp_err,           marker='+', c='black',      label='pp')
plt.xscale('log')
plt.yscale('log')
plt.xlim(0.1, 20)
plt.ylim(1E-6, 1E4)
plt.xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
plt.ylabel(r'$1/N_{\mathrm{evt}}\;dN/dp_{\mathrm{T}}$ (($\mathrm{GeV}/c$)$^{-1}$)')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)   # outside the axes, clear of the data
plt.show()


## 3. Nuclear Modification Factor $R_{AA}$

### 3.1–3.2 — Compute $R_{AA}$ and propagate uncertainties

$R_{AA}(p_T) = \dfrac{Y^{AA}(p_T) / \langle N_{coll}\rangle}{Y^{pp}(p_T)}$, with `at.uncertaintyCalculation` propagating the statistical (and pp systematic) uncertainties through that ratio.


In [ ]:
print(dictNColl)

RAA_0_5   = np.divide(hPbPb_0_5   / dictNColl['0-5'],   pp)
RAA_5_10  = np.divide(hPbPb_5_10  / dictNColl['5-10'],  pp)
RAA_10_20 = np.divide(hPbPb_10_20 / dictNColl['10-20'], pp)
RAA_20_30 = np.divide(hPbPb_20_30 / dictNColl['20-30'], pp)
RAA_30_40 = np.divide(hPbPb_30_40 / dictNColl['30-40'], pp)
RAA_40_50 = np.divide(hPbPb_40_50 / dictNColl['40-50'], pp)
RAA_50_60 = np.divide(hPbPb_50_60 / dictNColl['50-60'], pp)
RAA_60_70 = np.divide(hPbPb_60_70 / dictNColl['60-70'], pp)
RAA_70_80 = np.divide(hPbPb_70_80 / dictNColl['70-80'], pp)

RAA_0_5_err   = at.uncertaintyCalculation(hPbPb_0_5,   hPbPb_0_5_err,   dictNColl['0-5'],   pp, pp_err, dictNColl['pp'])
RAA_5_10_err  = at.uncertaintyCalculation(hPbPb_5_10,  hPbPb_5_10_err,  dictNColl['5-10'],  pp, pp_err, dictNColl['pp'])
RAA_10_20_err = at.uncertaintyCalculation(hPbPb_10_20, hPbPb_10_20_err, dictNColl['10-20'], pp, pp_err, dictNColl['pp'])
RAA_20_30_err = at.uncertaintyCalculation(hPbPb_20_30, hPbPb_20_30_err, dictNColl['20-30'], pp, pp_err, dictNColl['pp'])
RAA_30_40_err = at.uncertaintyCalculation(hPbPb_30_40, hPbPb_30_40_err, dictNColl['30-40'], pp, pp_err, dictNColl['pp'])
RAA_40_50_err = at.uncertaintyCalculation(hPbPb_40_50, hPbPb_40_50_err, dictNColl['40-50'], pp, pp_err, dictNColl['pp'])
RAA_50_60_err = at.uncertaintyCalculation(hPbPb_50_60, hPbPb_50_60_err, dictNColl['50-60'], pp, pp_err, dictNColl['pp'])
RAA_60_70_err = at.uncertaintyCalculation(hPbPb_60_70, hPbPb_60_70_err, dictNColl['60-70'], pp, pp_err, dictNColl['pp'])
RAA_70_80_err = at.uncertaintyCalculation(hPbPb_70_80, hPbPb_70_80_err, dictNColl['70-80'], pp, pp_err, dictNColl['pp'])

### 3.3 — Plot $R_{AA}$ vs. $p_T$

The plot stops at 10 GeV/$c$, as the $R_{CP}$ plot does: above it the three most peripheral classes have between 0 and 19 tracks per interval, so their points only scatter. The values above 10 GeV/$c$ are still computed and stay in the arrays.

Intervals with only one Pb–Pb track have a 100% statistical uncertainty and are not drawn (`MIN_TRACKS = 2`); they remain in the arrays. This removes the 70–80% point at 14–15 GeV/$c$ ($R_{AA} = 1.04$ from a single track) and a few single-track points at the highest $p_T$ in 60–70%.

In [ ]:
# An interval with a single Pb-Pb track (100% statistical uncertainty) is not drawn in the R_AA plots.
MIN_TRACKS = 2
nTracksPt = {cent: np.histogram(dictMomCent[cent], binsPt)[0] for cent in dictMomCent}
def drawable(values, cent):
    return np.where(nTracksPt[cent] >= MIN_TRACKS, values, np.nan)

plt.figure()
plt.errorbar(binsPtCenter, drawable(RAA_0_5, '0-5'),   xerr=x_bin_width, yerr=RAA_0_5_err,   marker='+', c='darkred',    label='0-5%')
plt.errorbar(binsPtCenter, drawable(RAA_5_10, '5-10'),  xerr=x_bin_width, yerr=RAA_5_10_err,  marker='+', c='red',        label='5-10%')
plt.errorbar(binsPtCenter, drawable(RAA_10_20, '10-20'), xerr=x_bin_width, yerr=RAA_10_20_err, marker='+', c='darkorange', label='10-20%')
plt.errorbar(binsPtCenter, drawable(RAA_20_30, '20-30'), xerr=x_bin_width, yerr=RAA_20_30_err, marker='+', c='gold',       label='20-30%')
plt.errorbar(binsPtCenter, drawable(RAA_30_40, '30-40'), xerr=x_bin_width, yerr=RAA_30_40_err, marker='+', c='yellowgreen',label='30-40%')
plt.errorbar(binsPtCenter, drawable(RAA_40_50, '40-50'), xerr=x_bin_width, yerr=RAA_40_50_err, marker='+', c='green',      label='40-50%')
plt.errorbar(binsPtCenter, drawable(RAA_50_60, '50-60'), xerr=x_bin_width, yerr=RAA_50_60_err, marker='+', c='deepskyblue',label='50-60%')
plt.errorbar(binsPtCenter, drawable(RAA_60_70, '60-70'), xerr=x_bin_width, yerr=RAA_60_70_err, marker='+', c='blue',       label='60-70%')
plt.errorbar(binsPtCenter, drawable(RAA_70_80, '70-80'), xerr=x_bin_width, yerr=RAA_70_80_err, marker='+', c='navy',       label='70-80%')
# pp reference: by definition R_AA(pp) = 1, plotted as a flat baseline
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='pp')
plt.ylim(0.01, 1.2)
plt.xlim(0, 10)   # shown up to 10 GeV/c: above it the peripheral classes have 0-19 tracks per interval
plt.xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
plt.ylabel(r'$R_{\mathrm{AA}}=\frac{1/N^{AA}_{\mathrm{evt}}\,dN^{AA}/dp_{\mathrm{T}}}{\langle N_{\mathrm{coll}}\rangle\,1/N^{pp}_{\mathrm{evt}}\,dN^{pp}/dp_{\mathrm{T}}}$')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)   # outside the axes, clear of the data
plt.savefig('RAA_compared.pdf', bbox_inches='tight')
plt.show()


### 3.4 — Example: read off a single $R_{AA}$ value

$R_{AA}$ in the bin $1.6 < p_T < 1.7$ GeV/c for the 70–80% centrality class.


In [ ]:
print(RAA_70_80[binsPt.index(1.6)])

### 3.5 — Comparison with ALICE's published $R_{AA}$

ALICE measured the same quantity with the full 2010 Pb–Pb data set, corrected for tracking efficiency, acceptance and secondary particles: **ALICE Collaboration, "Centrality dependence of charged particle production at large transverse momentum in Pb–Pb collisions at $\sqrt{s_{NN}}$ = 2.76 TeV", Phys. Lett. B 720 (2013) 52, arXiv:1208.2711**. The points below are the published values (HEPData record ins1127497, Tables 16, 20 and 24), with statistical and systematic uncertainties combined; ALICE's binning is the same as ours, so the two can be divided bin by bin.

Our points carry the statistical and pp-reference uncertainties; the box at $R_{AA} = 1$ is our $\langle N_{coll}\rangle$ normalisation uncertainty (ALICE, PRC 88 (2013) 044909). The lower panels show the ratio *this work / ALICE*: a ratio that is roughly constant in $p_T$ points to an overall normalisation difference — most likely the tracking efficiency, which our uncorrected Pb–Pb tracks do not include.

In [ ]:
# ALICE, PLB 720 (2013) 52 (arXiv:1208.2711): R_AA of primary charged particles, Pb-Pb 2.76 TeV, |eta| < 0.8
# HEPData ins1127497, Tables 16 (0-5%), 20 (30-40%), 24 (70-80%); err = statistical and systematic combined.
# Same bins as binsPt (0.15-15 GeV/c).
ALICE_RAA = {
    '0-5': dict(R=[0.1921, 0.2001, 0.2076, 0.2163, 0.2254, 0.2366, 0.248, 0.2589, 0.27, 0.281, 0.2927, 0.3038, 0.3137, 0.3255, 0.3355, 0.3462, 0.3552, 0.3673, 0.385, 0.3968, 0.4063, 0.4153, 0.4227, 0.4284, 0.4323, 0.4392, 0.4393, 0.43, 0.4158, 0.4037, 0.3804, 0.3532, 0.3248, 0.2983, 0.2791, 0.2565, 0.2339, 0.2032, 0.1721, 0.1494, 0.139, 0.1336, 0.1323, 0.1334, 0.1431, 0.1535, 0.1614, 0.1743, 0.1847, 0.1965, 0.2063],
           err=[0.0296, 0.026, 0.0249, 0.025, 0.0257, 0.0274, 0.0285, 0.0293, 0.0302, 0.0307, 0.0319, 0.0331, 0.03411, 0.03531, 0.03641, 0.03751, 0.03851, 0.0398, 0.04171, 0.04311, 0.04421, 0.04531, 0.04691, 0.04752, 0.04792, 0.04872, 0.04872, 0.04752, 0.04592, 0.04442, 0.04182, 0.03875, 0.03564, 0.03263, 0.03044, 0.02794, 0.02544, 0.02204, 0.01864, 0.01623, 0.01513, 0.01453, 0.01443, 0.01453, 0.01566, 0.0169, 0.01795, 0.01953, 0.02114, 0.02291, 0.02471]),
    '30-40': dict(R=[0.2815, 0.2828, 0.2901, 0.3001, 0.311, 0.3234, 0.3363, 0.3498, 0.3637, 0.3775, 0.3928, 0.4067, 0.419, 0.4329, 0.4469, 0.4592, 0.4712, 0.4847, 0.5067, 0.5232, 0.5354, 0.5438, 0.5534, 0.5603, 0.5661, 0.5771, 0.5776, 0.5667, 0.5561, 0.5495, 0.5303, 0.5065, 0.4829, 0.4598, 0.4468, 0.4281, 0.4091, 0.3797, 0.3519, 0.3293, 0.3183, 0.3137, 0.3136, 0.3151, 0.3304, 0.3458, 0.3671, 0.3771, 0.4014, 0.3985, 0.4212],
           err=[0.0418, 0.0358, 0.0349, 0.0347, 0.0358, 0.038, 0.0391, 0.0401, 0.0412, 0.0417, 0.0433, 0.04471, 0.04601, 0.04751, 0.04901, 0.05031, 0.05161, 0.05301, 0.05551, 0.05751, 0.05891, 0.06001, 0.06201, 0.06282, 0.06342, 0.06463, 0.06463, 0.06312, 0.06182, 0.06103, 0.05873, 0.05608, 0.05325, 0.05066, 0.04906, 0.04697, 0.04477, 0.04148, 0.03849, 0.03607, 0.03487, 0.03438, 0.0345, 0.03459, 0.03635, 0.03824, 0.04108, 0.04268, 0.04626, 0.04713, 0.05116]),
    '70-80': dict(R=[0.5797, 0.5626, 0.56, 0.5625, 0.5661, 0.5734, 0.5813, 0.5906, 0.6006, 0.6081, 0.6226, 0.6281, 0.6398, 0.6502, 0.6606, 0.6669, 0.6715, 0.6844, 0.6994, 0.7057, 0.7108, 0.7103, 0.7155, 0.717, 0.721, 0.726, 0.726, 0.7259, 0.7181, 0.7282, 0.7265, 0.7207, 0.6969, 0.6921, 0.6979, 0.7007, 0.6843, 0.6827, 0.6695, 0.674, 0.6648, 0.6487, 0.6739, 0.6513, 0.6479, 0.6902, 0.6656, 0.6051, 0.7718, 0.6905, 0.7634],
           err=[0.08904, 0.07671, 0.07541, 0.07301, 0.07402, 0.07631, 0.07682, 0.07722, 0.07772, 0.07713, 0.07893, 0.07934, 0.08075, 0.08205, 0.08327, 0.08408, 0.08449, 0.08606, 0.08818, 0.08926, 0.08996, 0.08999, 0.0918, 0.09195, 0.09245, 0.09301, 0.09292, 0.09251, 0.09142, 0.09254, 0.09215, 0.09131, 0.08841, 0.08762, 0.08826, 0.08849, 0.08642, 0.08613, 0.08461, 0.08528, 0.08432, 0.08264, 0.0862, 0.08324, 0.08366, 0.09058, 0.08996, 0.08583, 0.1126, 0.10911, 0.12759]),
}
NCOLL_REL_UNC = {'0-5': 0.113, '30-40': 0.095, '70-80': 0.066}   # ALICE PRC 88 (2013) 044909, Table 1

ours = {'0-5': (RAA_0_5, RAA_0_5_err), '30-40': (RAA_30_40, RAA_30_40_err), '70-80': (RAA_70_80, RAA_70_80_err)}
edges = np.asarray(binsPt)
from matplotlib.ticker import FixedLocator, NullFormatter
fig, axes = plt.subplots(2, 3, figsize=(10, 5), sharex=True, sharey='row',
                         gridspec_kw={'height_ratios': [2.2, 1], 'hspace': 0.05, 'wspace': 0.05})
for col, cent in enumerate(ours):
    R, R_err = ours[cent]
    R = drawable(R, cent)                     # single-track intervals not drawn
    A, A_err = np.asarray(ALICE_RAA[cent]['R']), np.asarray(ALICE_RAA[cent]['err'])
    ax, axr = axes[0, col], axes[1, col]
    for lo, hi, a, e in zip(edges[:-1], edges[1:], A, A_err):           # ALICE: boxes = total uncertainty
        ax.add_patch(plt.Rectangle((lo, a - e), hi - lo, 2 * e, color='0.75', lw=0))
    ax.plot(binsPtCenter, A, 's', color='0.35', ms=3.5, label='ALICE, PLB 720 (2013) 52')
    ax.errorbar(binsPtCenter, R, xerr=x_bin_width, yerr=R_err, fmt='o', ms=3.5, color='darkred', lw=1,
                label='this work (uncorrected tracks)')
    u = NCOLL_REL_UNC[cent]
    ax.add_patch(plt.Rectangle((0.16, 1 - u), 0.03, 2 * u, color='darkred', alpha=0.5))
    ax.axhline(1, color='black', ls='--', lw=0.8)
    ax.set_ylim(0, 1.4)
    ax.text(0.05, 0.94, f'{cent}%', transform=ax.transAxes, va='top', fontsize=11)
    ratio = R / A
    ratio_err = ratio * np.sqrt((R_err / R) ** 2 + (A_err / A) ** 2)
    axr.errorbar(binsPtCenter, ratio, xerr=x_bin_width, yerr=ratio_err, fmt='o', ms=3, color='darkred', lw=1)
    axr.axhline(1, color='black', ls='--', lw=0.8)
    axr.set_ylim(0.2, 1.6)
    axr.set_xscale('log')
    axr.set_xlim(0.15, 15)
    axr.xaxis.set_major_locator(FixedLocator([0.2, 0.5, 1, 2, 5, 10]))
    axr.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    axr.xaxis.set_minor_formatter(NullFormatter())
    axr.set_xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
    hi_pt = (binsPtCenter > 2) & (binsPtCenter < 10)
    print(f'{cent}%: mean ratio this work / ALICE for 2 < pT < 10 GeV/c = {np.mean(ratio[hi_pt]):.2f}')
axes[0, 0].set_ylabel(r'$R_{\mathrm{AA}}$')
axes[1, 0].set_ylabel('this work / ALICE')
axes[0, 0].legend(loc='center left', fontsize=8.5, frameon=False)
axes[0, 2].text(0.95, 0.94, r'Pb–Pb, $\sqrt{s_{NN}}$ = 2.76 TeV', transform=axes[0, 2].transAxes, ha='right', va='top', fontsize=9)
plt.savefig('RAA_vs_ALICE.pdf', bbox_inches='tight')
plt.show()

**Why the two disagree most at low $p_T$ in central collisions.** The ratio *this work / ALICE* is the part of the measurement that our raw track counts leave out. ALICE's pp reference and ALICE's published Pb–Pb result are both corrected; our Pb–Pb yield is not. So, to a good approximation,

$$\frac{R_{AA}^{\text{this work}}}{R_{AA}^{\text{ALICE}}} \approx \frac{N_{\text{reconstructed}}}{N_{\text{true}}} = \varepsilon(p_T,\ \text{centrality}),$$

the tracking efficiency × acceptance of the selected tracks (with smaller effects from secondary particles). Two features of $\varepsilon$ explain the plot:

1. **It falls at low $p_T$.** A slow particle curls tightly in the 0.5 T field (radius $R = p_T/(0.3B)$, i.e. 1.3 m at 0.2 GeV/$c$ against an outer TPC radius of about 2.5 m), loses more energy and scatters more in the detector material, so fewer of its tracks are found and pass the track-quality cuts. Above about 1 GeV/$c$ the efficiency reaches a plateau, which is why the ratio is flat there.
2. **It is lower in central collisions.** A 0–5% event has over 1,000 tracks in the TPC (Section 1.6). Clusters from neighbouring tracks overlap and are shared, so track finding loses more tracks — and low-$p_T$ tracks, which are strongly curved and leave wider clusters, are hit hardest by overlaps. A 70–80% event has a few tens of tracks, so its efficiency is close to the low-occupancy value.

Hence the ratio is about 0.76 on the plateau in 0–5%, falls to about 0.3 at 0.2 GeV/$c$, and is close to 1 on the plateau in 70–80%. The physics shape of $R_{AA}$ — the maximum near 2 GeV/$c$, the minimum at 6–7 GeV/$c$ — is unaffected, because the efficiency is nearly flat where those features are.


## 4. Central-to-Peripheral Ratio $R_{CP}$

$R_{CP}$ compares a central (more head-on) collision class to a peripheral (more glancing) collision class, instead of comparing to pp. Using 70–80% as the peripheral reference removes some of the need for an independent pp measurement and cancels common systematic uncertainties between the two Pb–Pb classes:

$$
R_{CP}(p_T) = \frac{ \langle N_{coll}\rangle^{peripheral} / N_{evt}^{central} \; dN^{central}/dp_T }
                    { \langle N_{coll}\rangle^{central} / N_{evt}^{peripheral} \; dN^{peripheral}/dp_T }
$$

i.e. each centrality's yield (already normalised per event and bin width, as in `hPbPb_*`) divided by its own $N_{coll}$, relative to the same quantity for the 70–80% peripheral bin.


In [ ]:
peripheral_label = '70-80'
hPbPb_peripheral     = hPbPb_70_80
hPbPb_peripheral_err = hPbPb_70_80_err
NColl_peripheral      = dictNColl['70-80']

RCP_0_5   = np.divide(hPbPb_0_5   / dictNColl['0-5'],   hPbPb_peripheral / NColl_peripheral)
RCP_5_10  = np.divide(hPbPb_5_10  / dictNColl['5-10'],  hPbPb_peripheral / NColl_peripheral)
RCP_10_20 = np.divide(hPbPb_10_20 / dictNColl['10-20'], hPbPb_peripheral / NColl_peripheral)
RCP_20_30 = np.divide(hPbPb_20_30 / dictNColl['20-30'], hPbPb_peripheral / NColl_peripheral)
RCP_30_40 = np.divide(hPbPb_30_40 / dictNColl['30-40'], hPbPb_peripheral / NColl_peripheral)
RCP_40_50 = np.divide(hPbPb_40_50 / dictNColl['40-50'], hPbPb_peripheral / NColl_peripheral)
RCP_50_60 = np.divide(hPbPb_50_60 / dictNColl['50-60'], hPbPb_peripheral / NColl_peripheral)
RCP_60_70 = np.divide(hPbPb_60_70 / dictNColl['60-70'], hPbPb_peripheral / NColl_peripheral)

RCP_0_5_err   = at.uncertaintyCalculation(hPbPb_0_5,   hPbPb_0_5_err,   dictNColl['0-5'],   hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_5_10_err  = at.uncertaintyCalculation(hPbPb_5_10,  hPbPb_5_10_err,  dictNColl['5-10'],  hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_10_20_err = at.uncertaintyCalculation(hPbPb_10_20, hPbPb_10_20_err, dictNColl['10-20'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_20_30_err = at.uncertaintyCalculation(hPbPb_20_30, hPbPb_20_30_err, dictNColl['20-30'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_30_40_err = at.uncertaintyCalculation(hPbPb_30_40, hPbPb_30_40_err, dictNColl['30-40'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_40_50_err = at.uncertaintyCalculation(hPbPb_40_50, hPbPb_40_50_err, dictNColl['40-50'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_50_60_err = at.uncertaintyCalculation(hPbPb_50_60, hPbPb_50_60_err, dictNColl['50-60'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)
RCP_60_70_err = at.uncertaintyCalculation(hPbPb_60_70, hPbPb_60_70_err, dictNColl['60-70'], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral)

plt.figure()
plt.errorbar(binsPtCenter, RCP_0_5,   xerr=x_bin_width, yerr=RCP_0_5_err,   marker='+', c='darkred',    label='0-5%')
plt.errorbar(binsPtCenter, RCP_5_10,  xerr=x_bin_width, yerr=RCP_5_10_err,  marker='+', c='red',        label='5-10%')
plt.errorbar(binsPtCenter, RCP_10_20, xerr=x_bin_width, yerr=RCP_10_20_err, marker='+', c='darkorange', label='10-20%')
plt.errorbar(binsPtCenter, RCP_20_30, xerr=x_bin_width, yerr=RCP_20_30_err, marker='+', c='gold',       label='20-30%')
plt.errorbar(binsPtCenter, RCP_30_40, xerr=x_bin_width, yerr=RCP_30_40_err, marker='+', c='yellowgreen',label='30-40%')
plt.errorbar(binsPtCenter, RCP_40_50, xerr=x_bin_width, yerr=RCP_40_50_err, marker='+', c='green',      label='40-50%')
# Peripheral reference: by definition R_CP(70-80%) = 1, plotted as a flat baseline
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='70-80% (reference)')
plt.ylim(0.01, 2.0)
plt.xlim(0, 10)   # shown up to 10 GeV/c
plt.xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
plt.ylabel(r'$R_{\mathrm{CP}}=\frac{1/N^{central}_{\mathrm{evt}}\,dN^{central}/dp_{\mathrm{T}} \,/\, N_{\mathrm{coll}}^{central}}'
           r'{1/N^{peripheral}_{\mathrm{evt}}\,dN^{peripheral}/dp_{\mathrm{T}} \,/\, N_{\mathrm{coll}}^{peripheral}}$')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)   # outside the axes, clear of the data
plt.savefig('RCP_compared.pdf', bbox_inches='tight')
plt.show()

### 4.1 — Comparison with ALICE: $R_{CP}$

ALICE did not publish $R_{CP}$ for these classes, but it published the corrected $p_T$ spectra of every class (same paper, PLB 720 (2013) 52; HEPData ins1127497, Tables 1, 5 and 9). Dividing those spectra with the same $\langle N_{coll}\rangle$ values as above gives ALICE's $R_{CP}$ in exactly our definition and bins. ALICE's spectrum uncertainties combine statistical and systematic parts; much of the systematic part is common to both classes and would cancel in the ratio, so the grey bands are an upper limit on ALICE's $R_{CP}$ uncertainty.

The efficiency argument from Section 3.5 predicts what to expect: the pp reference cancels in $R_{CP}$, but the efficiency does not, because it depends on centrality. The ratio *this work / ALICE* should be $\varepsilon_{\text{central}}/\varepsilon_{70\text{–}80\%}$ — about $0.76/0.97 \approx 0.8$ for 0–5% on the plateau, closer to 1 for 30–40%, and lower at low $p_T$.

In [ ]:
# ALICE, PLB 720 (2013) 52: corrected spectra 1/N_evt 1/(2 pi pT) d2N/(deta dpT), (GeV/c)^-2, |eta| < 0.8
# HEPData ins1127497, Tables 1 (0-5%), 5 (30-40%), 9 (70-80%); err = statistical and systematic combined.
# Same bins as binsPt (0.15-15 GeV/c). The invariant-yield factor 1/(2 pi pT) cancels in the ratio.
ALICE_SPECTRA = {
    '0-5': dict(Y=[1810, 1509, 1226, 985.5, 791.4, 644.1, 526.6, 433.1, 358.5, 298.5, 250.2, 210.9, 178.4, 151.4, 129.3, 110.9, 94.85, 75.5, 56.54, 42.52, 31.96, 24.34, 18.52, 14.29, 11.03, 8.641, 6.72, 4.683, 2.865, 1.774, 1.107, 0.6974, 0.4408, 0.2836, 0.1851, 0.1222, 0.08176, 0.0425, 0.01805, 0.008502, 0.004461, 0.002534, 0.001543, 0.0008184, 0.0003937, 0.0002071, 0.0001145, 6.889e-05, 4.269e-05, 2.767e-05, 1.833e-05],
           err=[243, 171, 121, 89.3, 67.6, 54, 43.8, 35.8, 29.6, 24.5, 20.5, 17.2, 14.5, 12.3, 10.5, 9.002, 7.702, 6.121, 4.591, 3.471, 2.61, 2, 1.57, 1.21, 0.9302, 0.7293, 0.5672, 0.3931, 0.24, 0.148, 0.09202, 0.05781, 0.03642, 0.02341, 0.01521, 0.01001, 0.006665, 0.003454, 0.001462, 0.00069, 0.0003627, 0.0002055, 0.0001255, 6.651e-05, 3.206e-05, 1.693e-05, 9.519e-06, 5.752e-06, 3.612e-06, 2.372e-06, 1.607e-06]),
    '30-40': dict(Y=[506, 406.7, 326.6, 260.7, 208.2, 167.9, 136.2, 111.6, 92.13, 76.48, 64.04, 53.84, 45.45, 38.41, 32.85, 28.05, 23.99, 19, 14.19, 10.7, 8.031, 6.08, 4.624, 3.566, 2.754, 2.165, 1.685, 1.177, 0.7309, 0.4604, 0.2943, 0.1907, 0.125, 0.08338, 0.05653, 0.0389, 0.02727, 0.01515, 0.007038, 0.003573, 0.001949, 0.001135, 0.0006974, 0.0003686, 0.0001734, 8.897e-05, 4.967e-05, 2.843e-05, 1.77e-05, 1.071e-05, 7.14e-06],
           err=[64.5, 44.6, 32.4, 23.6, 18.1, 14.4, 11.6, 9.401, 7.771, 6.42, 5.36, 4.49, 3.78, 3.19, 2.72, 2.321, 1.98, 1.57, 1.18, 0.8901, 0.6701, 0.5091, 0.3981, 0.3061, 0.2371, 0.1861, 0.1441, 0.1, 0.06201, 0.03891, 0.02481, 0.016, 0.01051, 0.006955, 0.004694, 0.003223, 0.002252, 0.001241, 0.0005788, 0.0002935, 0.0001604, 9.326e-05, 5.752e-05, 3.04e-05, 1.438e-05, 7.385e-06, 4.166e-06, 2.41e-06, 1.526e-06, 9.396e-07, 6.405e-07]),
    '70-80': dict(Y=[49.67, 38.56, 30.05, 23.29, 18.06, 14.19, 11.22, 8.981, 7.25, 5.871, 4.839, 3.963, 3.308, 2.749, 2.314, 1.941, 1.63, 1.279, 0.9336, 0.6875, 0.5082, 0.3785, 0.285, 0.2175, 0.1672, 0.1298, 0.1009, 0.07187, 0.04498, 0.02908, 0.01921, 0.01293, 0.008598, 0.005982, 0.004208, 0.003034, 0.002174, 0.001298, 0.0006382, 0.0003486, 0.000194, 0.0001119, 7.143e-05, 3.632e-05, 1.621e-05, 8.462e-06, 4.292e-06, 2.174e-06, 1.622e-06, 8.841e-07, 6.167e-07],
           err=[6.633, 4.661, 3.501, 2.521, 1.92, 1.511, 1.19, 0.9453, 0.7603, 0.6133, 0.5053, 0.4122, 0.3432, 0.2853, 0.2403, 0.2012, 0.1682, 0.1321, 0.09682, 0.07157, 0.05295, 0.03956, 0.03024, 0.02305, 0.01774, 0.01374, 0.01073, 0.007562, 0.004728, 0.003045, 0.002014, 0.001353, 0.0008985, 0.0006229, 0.000438, 0.0003153, 0.0002257, 0.0001344, 6.621e-05, 3.625e-05, 2.021e-05, 1.173e-05, 7.518e-06, 3.819e-06, 1.725e-06, 9.219e-07, 4.865e-07, 2.619e-07, 2.008e-07, 1.209e-07, 8.996e-08]),
}

def alice_rcp(cent, ref='70-80'):
    c, p = ALICE_SPECTRA[cent], ALICE_SPECTRA[ref]
    Yc, Ec, Yp, Ep = map(np.asarray, (c['Y'], c['err'], p['Y'], p['err']))
    R = (Yc / dictNColl[cent]) / (Yp / dictNColl[ref])
    return R, R * np.sqrt((Ec / Yc) ** 2 + (Ep / Yp) ** 2)

from matplotlib.ticker import FixedLocator, NullFormatter
ours_rcp = {'0-5': (RCP_0_5, RCP_0_5_err), '30-40': (RCP_30_40, RCP_30_40_err)}
edges = np.asarray(binsPt)
fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex=True, sharey='row',
                         gridspec_kw={'height_ratios': [2.2, 1], 'hspace': 0.05, 'wspace': 0.05})
for col, cent in enumerate(ours_rcp):
    R, R_err = ours_rcp[cent]
    A, A_err = alice_rcp(cent)
    ax, axr = axes[0, col], axes[1, col]
    for lo, hi, a, e in zip(edges[:-1], edges[1:], A, A_err):           # ALICE: boxes = uncertainty
        ax.add_patch(plt.Rectangle((lo, a - e), hi - lo, 2 * e, color='0.75', lw=0))
    ax.plot(binsPtCenter, A, 's', color='0.35', ms=3.5, label='ALICE spectra, PLB 720 (2013) 52')
    ax.errorbar(binsPtCenter, R, xerr=x_bin_width, yerr=R_err, fmt='o', ms=3.5, color='darkblue', lw=1,
                label='this work (uncorrected tracks)')
    ax.axhline(1, color='black', ls='--', lw=0.8)
    ax.set_ylim(0, 1.4)
    ax.text(0.05, 0.94, f'{cent}% / 70–80%', transform=ax.transAxes, va='top', fontsize=11)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = R / A
        ratio_err = ratio * np.sqrt((R_err / R) ** 2 + (A_err / A) ** 2)
    axr.errorbar(binsPtCenter, ratio, xerr=x_bin_width, yerr=ratio_err, fmt='o', ms=3, color='darkblue', lw=1)
    axr.axhline(1, color='black', ls='--', lw=0.8)
    axr.set_ylim(0.2, 1.6)
    axr.set_xscale('log')
    axr.set_xlim(0.15, 10)
    axr.xaxis.set_major_locator(FixedLocator([0.2, 0.5, 1, 2, 5, 10]))
    axr.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    axr.xaxis.set_minor_formatter(NullFormatter())
    axr.set_xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
    plateau = (binsPtCenter > 2) & (binsPtCenter < 10)
    print(f'{cent}% / 70-80%: mean ratio this work / ALICE for 2 < pT < 10 GeV/c = {np.mean(ratio[plateau]):.2f}')
axes[0, 0].set_ylabel(r'$R_{\mathrm{CP}}$')
axes[1, 0].set_ylabel('this work / ALICE')
axes[0, 0].legend(loc='center left', fontsize=8.5, frameon=False)
axes[0, 1].text(0.95, 0.05, r'Pb–Pb, $\sqrt{s_{NN}}$ = 2.76 TeV', transform=axes[0, 1].transAxes, ha='right', va='bottom', fontsize=9)
plt.savefig('RCP_vs_ALICE.pdf', bbox_inches='tight')
plt.show()


---
# Part B — A species-agnostic $R_{AA}$/$R_{CP}$ engine

Look closely at Sections 3 and 4 above: **the only things that differ between an $R_{AA}$ calculation and an $R_{CP}$ calculation are which "reference" spectrum and which $N_{coll}$ you divide by.** Nothing in the formula or its uncertainty propagation refers to charged hadrons specifically. That means we can lift it into one small function that works for *any* particle species, as long as you can hand it a yield-vs-$p_T$ spectrum and its $N_{coll}$.

`compute_modification_factor` below is exactly the calculation from Sections 3.1–3.2 and 4, just parametrised:
- pass the pp spectrum and `dictNColl['pp']` (=1) → you get $R_{AA}$
- pass a peripheral Pb–Pb spectrum and its own $N_{coll}$ → you get $R_{CP}$


In [ ]:
def compute_modification_factor(yield_AA, yield_AA_err, Ncoll_AA,
                                  yield_ref, yield_ref_err, Ncoll_ref=1.0):
    """
    Generic nuclear modification factor, R = (yield_AA / Ncoll_AA) / (yield_ref / Ncoll_ref).

    With yield_ref = the pp spectrum and Ncoll_ref = dictNColl['pp'] (= 1), this is R_AA
    (Section 3). With yield_ref = a peripheral Pb-Pb spectrum and Ncoll_ref = that
    centrality's N_coll, this is R_CP (Section 4). Species-agnostic: yield_AA/yield_ref
    can be a charged-track spectrum, a J/psi spectrum, or anything else measured the
    same way in both the numerator and denominator sample.
    """
    R = np.divide(yield_AA / Ncoll_AA, yield_ref / Ncoll_ref)
    R_err = at.uncertaintyCalculation(yield_AA, yield_AA_err, Ncoll_AA,
                                       yield_ref, yield_ref_err, Ncoll_ref)
    return R, R_err


### Sanity check: does the generic function reproduce Part A exactly?

This isn't optional — before trusting the generic engine for a new particle species, we confirm it reproduces the original, hand-written $R_{AA}$ and $R_{CP}$ arrays from Part A bit-for-bit (up to floating-point rounding).


In [ ]:
RAA_0_5_generic, RAA_0_5_generic_err = compute_modification_factor(
    hPbPb_0_5, hPbPb_0_5_err, dictNColl['0-5'], pp, pp_err, dictNColl['pp']
)
RCP_60_70_generic, RCP_60_70_generic_err = compute_modification_factor(
    hPbPb_60_70, hPbPb_60_70_err, dictNColl['60-70'],
    hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral
)

print("R_AA(0-5%) matches Part A:", np.allclose(RAA_0_5_generic, RAA_0_5, equal_nan=True))
print("R_AA(0-5%) error matches Part A:", np.allclose(RAA_0_5_generic_err, RAA_0_5_err, equal_nan=True))
print("R_CP(60-70%) matches Part A:", np.allclose(RCP_60_70_generic, RCP_60_70, equal_nan=True))
print("R_CP(60-70%) error matches Part A:", np.allclose(RCP_60_70_generic_err, RCP_60_70_err, equal_nan=True))


### Applying it across all centralities in one loop

The per-centrality variables from Part A (`hPbPb_0_5`, `hPbPb_5_10`, ...) are collected into a dictionary here purely for convenience — they are *not* recomputed, just referenced — and then `compute_modification_factor` is applied to every centrality class in a loop instead of nine repeated lines.


In [ ]:
hPbPb_by_cent = {
    '0-5':   (hPbPb_0_5,   hPbPb_0_5_err),
    '5-10':  (hPbPb_5_10,  hPbPb_5_10_err),
    '10-20': (hPbPb_10_20, hPbPb_10_20_err),
    '20-30': (hPbPb_20_30, hPbPb_20_30_err),
    '30-40': (hPbPb_30_40, hPbPb_30_40_err),
    '40-50': (hPbPb_40_50, hPbPb_40_50_err),
    '50-60': (hPbPb_50_60, hPbPb_50_60_err),
    '60-70': (hPbPb_60_70, hPbPb_60_70_err),
    '70-80': (hPbPb_70_80, hPbPb_70_80_err),
}

RAA_all_generic = {}
for cent, (yld, yld_err) in hPbPb_by_cent.items():
    RAA_all_generic[cent] = compute_modification_factor(
        yld, yld_err, dictNColl[cent], pp, pp_err, dictNColl['pp']
    )

# Spot-check against the named Part A variables again
print("R_AA(70-80%) matches Part A:", np.allclose(RAA_all_generic['70-80'][0], RAA_70_80, equal_nan=True))


### Plot: $R_{AA}$ from the generic loop

Same data, same numbers as Part A, Section 3.3 — just produced by the loop above instead of nine repeated blocks. Compare this to the Section 3.3 plot: they should be identical.


In [ ]:
cent_colors = {
    '0-5': 'darkred', '5-10': 'red', '10-20': 'darkorange', '20-30': 'gold',
    '30-40': 'yellowgreen', '40-50': 'green', '50-60': 'deepskyblue',
    '60-70': 'blue', '70-80': 'navy',
}

plt.figure()
for cent, color in cent_colors.items():
    R, R_err = RAA_all_generic[cent]
    plt.errorbar(binsPtCenter, drawable(R, cent), xerr=x_bin_width, yerr=R_err, marker='+', c=color, label=f'{cent}%')
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='pp')
plt.ylim(0.01, 1.2)
plt.xlim(0, 10)   # shown up to 10 GeV/c: above it the peripheral classes have 0-19 tracks per interval
plt.xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
plt.ylabel(r'$R_{\mathrm{AA}}$ (from the generic loop)')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)   # outside the axes, clear of the data
plt.show()


### $R_{CP}$ from the same generic engine

The same loop pattern, with the reference swapped from pp to the 70–80% peripheral class — reproducing Part A, Section 4.


In [ ]:
RCP_all_generic = {}
for cent in ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50', '50-60', '60-70']:
    yld, yld_err = hPbPb_by_cent[cent]
    RCP_all_generic[cent] = compute_modification_factor(
        yld, yld_err, dictNColl[cent], hPbPb_peripheral, hPbPb_peripheral_err, NColl_peripheral
    )

print("R_CP(60-70%) matches Part A:", np.allclose(RCP_all_generic['60-70'][0], RCP_60_70, equal_nan=True))

plt.figure()
for cent in ['0-5', '5-10', '10-20', '20-30', '30-40', '40-50']:   # 50-70% not drawn
    R, R_err = RCP_all_generic[cent]
    plt.errorbar(binsPtCenter, R, xerr=x_bin_width, yerr=R_err, marker='+', c=cent_colors[cent], label=f'{cent}%')
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='70-80% (reference)')
plt.ylim(0.01, 2.0)
plt.xlim(0, 10)   # shown up to 10 GeV/c
plt.xlabel(r'$p_{\mathrm{T}}$ ($\mathrm{GeV}/c$)', fontsize=14)
plt.ylabel(r'$R_{\mathrm{CP}}$ (from the generic loop)')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)   # outside the axes, clear of the data
plt.show()

---
# Part C — The ALICE J/$\psi$ masterclass, step by step

### C.1 — The physics in the manual

**The J/$\psi$.** A meson made of a charm quark and a charm antiquark ($c\bar c$), with mass $m_{J/\psi} = 3.097$ GeV/$c^2$. It was discovered in 1974 by two teams at once, led by Burton Richter and Samuel Ting (Nobel Prize 1976); Ting wanted to call it J and Richter $\psi$, hence the double name. Its discovery, the "November revolution", established a fourth quark, charm. It lives about $7\times10^{-21}$ s and decays, among other ways, to an electron–positron pair or a muon pair. The masterclass looks for

$$J/\psi \to e^+ e^- .$$

**Natural units.** With $c = 1$, energy, momentum and mass are all measured in GeV (momentum in GeV/$c$, mass in GeV/$c^2$, written as GeV below).

**Step 1 — energy and momentum.** Classically, $E_{kin} = \tfrac12 m v^2$ and $p = m v$. Multiplying the first by $m/m$:

$$E_{kin} = \frac{m^2 v^2}{2m} = \frac{p^2}{2m}.$$

This only holds for $v \ll c$. In ALICE the particles move at nearly the speed of light, and special relativity replaces it with

$$E^2 = (mc^2)^2 + (pc)^2 \quad\xrightarrow{\;c\,=\,1\;}\quad E = \sqrt{p^2 + m^2}.$$

This is the line `e = TMath::Sqrt(pSquared + massSquared)` in `LoadTracksFromTree()`.

**Step 2 — invariant mass.** For any system of particles, the combination

$$M^2 = E_{tot}^2 - |\vec p_{tot}|^2$$

is the same in every reference frame (it is "invariant"). In the rest frame of a decaying particle, $\vec p_{tot} = 0$ and $E_{tot}$ is its rest energy, so $M$ is its mass. Energy and momentum are conserved in the decay $J/\psi \to e^+e^-$, so $E_{tot} = E_1 + E_2$ and $\vec p_{tot} = \vec p_1 + \vec p_2$, and the mass of the parent follows from the two daughters alone:

$$M = \sqrt{(E_1+E_2)^2 - |\vec p_1 + \vec p_2|^2}.$$

This is the formula in `FillInvariantMassHistos()`. Expanding it with $E_i^2 - |\vec p_i|^2 = m_i^2$:

$$M^2 = m_1^2 + m_2^2 + 2\left(E_1E_2 - \vec p_1\cdot\vec p_2\right) \;\approx\; 2\,p_1 p_2\,(1 - \cos\theta_{12}) \quad (m_1, m_2 \approx 0),$$

so a large mass needs fast tracks with a wide opening angle $\theta_{12}$.

**Step 3 — which mass to use for a track.** The detector measures momentum, not mass. The program decides from the energy loss: a track with dE/dx $> 62$ is taken to be an electron, $m^2 = 0$ ($m_e = 0.511$ MeV, so $m_e^2 = 2.6\times10^{-7}$ GeV$^2$ is negligible); any other track is taken to be a pion, $m^2 = 0.019$ GeV$^2$ ($m_\pi = 0.1396$ GeV, $m_\pi^2 = 0.0195$ GeV$^2$).

**Step 4 — the momentum of a J/$\psi$ electron.** In the J/$\psi$ rest frame the two daughters fly apart back to back with $\vec p_1 = -\vec p_2$. They have equal masses, so equal energies, and $M = E_1 + E_2 = 2E$. With $m_e \approx 0$, $|\vec p| = E$:

$$|\vec p_{e}| = \frac{m_{J/\psi}}{2} = \frac{3.097}{2} \approx 1.55 \text{ GeV}/c .$$

In the laboratory the J/$\psi$ is usually moving, which spreads the two momenta around this value. This is the manual's hint for step 3: electrons from a J/$\psi$ carry a large momentum, about half the J/$\psi$ mass, so a minimum momentum removes many electrons from other sources, such as photon conversions $\gamma \to e^+e^-$.

**Step 5 — particle identification.** A charged particle ionises the gas of ALICE's Time Projection Chamber (TPC). The energy it loses per unit length, dE/dx, depends on its velocity, so at a given momentum different particle species lose different amounts: plotted against momentum they form separate bands. Electrons, the lightest, are already close to the speed of light and form a nearly flat band above the pions.

**Step 6 — signal and background.** Most $e^-e^+$ pairs are random combinations of unrelated tracks. A J/$\psi$ always gives one $e^-$ and one $e^+$, so same-charge pairs can never contain one; for random pairing they estimate the random opposite-charge pairs. Inside a mass window:

$$S = N_{e^-e^+} - \left(N_{e^-e^-} + N_{e^+e^+}\right), \qquad B = N_{e^-e^-} + N_{e^+e^+}, \qquad \text{significance} = \frac{S}{\sqrt{S + B}} .$$

$\sqrt{S+B} = \sqrt{N_{e^-e^+}}$ is the size of the random (Poisson) fluctuation of the opposite-charge count, so the significance says by how many such fluctuations the signal stands out.

### C.2 — Configuration

In [ ]:
import os
import re
# Part C configuration. JPSI_DATA_DIR points at the 'data/events' folder of the ALICE J/psi masterclass
# (JpsiMasterclass.zip). The masterclass start menu offers two datasets, and alice_vsd(dataset) in
# masterclass.C opens  data/events/events_<dataset>.root :
#   dataset 1 -> "p-p collision"    (proton-proton, sqrt(s) = 7 TeV)
#   dataset 2 -> "p-Pb collision"   (proton-lead)
MASTERCLASS_DIR = 'JpsiMasterclass'                         # the unzipped JpsiMasterclass folder
JPSI_DATA_DIR = os.path.join(MASTERCLASS_DIR, 'data', 'events')
DATASETS = {1: 'pp', 2: 'pPb'}
DATASET_TITLE = {1: 'pp, $\\sqrt{s}$ = 7 TeV', 2: 'p–Pb'}
SAVE_FILE = 'masterclass_notebook.save'   # the C program appends to masterclass.save; a separate name keeps the zip's copy intact

try:
    import uproot  # reads ROOT files in pure Python
except ImportError:
    # not installed in this Jupyter environment: install it into the kernel's own Python, then import it
    import subprocess, sys
    print("Installing uproot (needed to read the ROOT event files) ...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'uproot'])
    import importlib
    importlib.invalidate_caches()
    import uproot
print(f"uproot {uproot.__version__}")

### C.3 — Reading the events

In [ ]:
_event_cache = {}

def read_vsd_events(dataset):
    """
    Read every event of  data/events/events_<dataset>.root  the way TVSDReader does.

    masterclass.C (TVSDReader constructor): the events are the keys of the file whose name matches the
    regular expression "Event\\d+", in the order of the file's key list. GotoEvent() then reads that event
    directory's track tree; LoadTracksFromTree() takes from each track
        px, py, pz = track->GetMomentum()   (Float_t)
        dedx       = track->GetStatus()     (the TPC dE/dx is stored in the fStatus field)
        charge     = track->GetCharge()
    An event directory without a track tree simply has no tracks (it still counts as an event).
    Reading ~6,000 events takes about a minute; the result is cached.
    """
    if dataset in _event_cache:
        return _event_cache[dataset]
    f = uproot.open(os.path.join(JPSI_DATA_DIR, f'events_{dataset}.root'))
    names = [k for k in f.keys() if '/' not in k and re.search(r'Event\d+', k)]
    events = []
    for name in names:
        try:
            tree = f[name.split(';')[0] + '/RecTracks']
        except KeyError:
            tree = None
        if tree is None:
            events.append({'px': np.zeros(0, np.float32), 'py': np.zeros(0, np.float32), 'pz': np.zeros(0, np.float32),
                           'dedx': np.zeros(0, np.float32), 'charge': np.zeros(0, np.int32)})
            continue
        a = tree.arrays(['fP.fX', 'fP.fY', 'fP.fZ', 'fStatus', 'fSign'], library='np')
        events.append({'px': a['fP.fX'].astype(np.float32), 'py': a['fP.fY'].astype(np.float32),
                       'pz': a['fP.fZ'].astype(np.float32),
                       'dedx': a['fStatus'].astype(np.float32),        # Float_t dedx = track->GetStatus();
                       'charge': a['fSign'].astype(np.int32)})
    _event_cache[dataset] = events
    return events

for ds in DATASETS:
    evs = read_vsd_events(ds)
    print(f"dataset {ds} ({DATASETS[ds]}): {len(evs):,} events, {sum(len(e['dedx']) for e in evs):,} reconstructed tracks")

In [ ]:
# ROOT-style fixed-bin histogram, so binning, under/overflow and Integral() behave exactly as TH1F in masterclass.C.
class Hist1D:
    """TH1F with nbins fixed bins on [xmin, xmax). counts[0] is the underflow bin and counts[nbins+1] the overflow
    bin, as in ROOT; FindBin() returns ROOT's 1-based bin number and Integral(b1, b2) includes both end bins."""
    def __init__(self, nbins, xmin, xmax):
        self.nbins, self.xmin, self.xmax = nbins, float(xmin), float(xmax)
        self.counts = np.zeros(nbins + 2)

    def FindBin(self, x):
        x = np.asarray(x, dtype=np.float64)
        b = 1 + np.floor(self.nbins * (x - self.xmin) / (self.xmax - self.xmin))
        b = np.where(x < self.xmin, 0, b)
        b = np.where(~(x < self.xmax), self.nbins + 1, b)   # x >= xmax, and NaN, go to the overflow bin, as in TAxis
        return b.astype(int)

    def Fill(self, x):
        np.add.at(self.counts, self.FindBin(x), 1.0)

    def Add(self, other, c=1.0):
        self.counts += c * other.counts

    def Integral(self, b1, b2):
        return float(np.sum(self.counts[b1:b2 + 1]))

    def Reset(self):
        self.counts[:] = 0.0

    def edges(self):
        return np.linspace(self.xmin, self.xmax, self.nbins + 1)

    def contents(self):
        """Bin contents without under/overflow (what the histogram shows)."""
        return self.counts[1:-1]

In [ ]:
F32 = np.float32

class TVSDReader:
    """
    Python port of the analysis in masterclass.C (class TVSDReader). Method and member names are the C++ ones,
    so each step can be compared line by line with the original. The 3D event display, the GUI widgets and the
    ROOT drawing calls are not ported; plotting is done separately with matplotlib.

    Arithmetic follows the C++ types: momenta, energies and masses are Float_t (32-bit) and square roots are
    TMath::Sqrt (64-bit) of the 32-bit argument, rounded back to Float_t when stored.

    Two deliberate differences from the C++, neither of which changes any J/psi number:
      * The C++ stores each selected track's (p, dE/dx) at its position in the full track list but then reads
        only the first nSelectedTracks positions, so with track cuts applied the dE/dx-vs-p plot shows some of the
        rejected tracks as zeros and misses some selected ones. Here the selected tracks are stored in order.
      * record_pairs=True (an extension, not in masterclass.C) keeps every pair's mass and transverse momentum
        (an option for further analyses; not used in this notebook).
    """
    ELECTRON_DEDX = F32(62)        # Float_t massSquared = dedx>62 ? 0. : 0.019;
    PION_MASS_SQUARED = F32(0.019)

    def __init__(self, dataset, record_pairs=False):
        # alice_vsd(dataset): vsd_file_name = Form("data/events/events_%i.root", dataset)
        self.dataset = dataset
        self.events = read_vsd_events(dataset)
        self.fMaxEv = len(self.events)
        self.fCurEv = -1
        self.fLoadTracks = False
        self.fApplyTrackCuts = False
        self.fEnableQuickAnalysis = False
        self.p1 = self.p2 = self.de1 = self.de2 = F32(0)
        self.minm = self.maxm = F32(0)
        self.epsilon = F32(0.01)                # epsilon(0.01)
        self.nEvents = 0
        self.numJpsi = 0                        # Int_t
        self.bg = 0                             # Int_t
        self.significance = F32(0)
        self.sb = F32(0)
        # histograms as booked in the constructor
        self.gMinvHist = Hist1D(60, 0., 6.)     # "Invariant Mass Distribution: Electrons + Positrons"
        self.geeHist = Hist1D(60, 0., 6.)       # "Electrons + Electrons"
        self.gppHist = Hist1D(60, 0., 6.)       # "Positrons + Positrons"
        self.gJpsiHist = Hist1D(60, 0., 6.)     # "Difference: EP - (EE + PP)"
        # gEnergyLoss: 200 logarithmic momentum bins from 0.1 to 20 GeV/c, 120 dE/dx bins from 20 to 140
        expMax = F32(np.log(np.float64(F32(20.) / F32(0.1))))       # Float_t expMax = TMath::Log(lastX/firstX)
        self.pid_xedges = np.array([F32(0.1) * np.exp(np.float64(F32(expMax / F32(200)) * F32(i))) for i in range(201)],
                                   dtype=np.float32)
        self.pid_yedges = np.array([20. + i * (140. - 20.) / 120 for i in range(121)], dtype=np.float32)
        self.gEnergyLoss = np.zeros((200, 120))
        # per-event track arrays (el[100][4], po[100][4], gArrP, gArrdEdx in the C++)
        self.el = np.zeros((0, 4), np.float32)
        self.po = np.zeros((0, 4), np.float32)
        self.gArrP = np.zeros(0, np.float32)
        self.gArrdEdx = np.zeros(0, np.float32)
        self.nSelectedTracks = 0
        self.record_pairs = record_pairs
        self.pair_log = {'unlike': [], 'like': []}
        self.last_event_pairs = None
        # StudentSet(); GotoEvent(0);
        self.GotoEvent(0)

    # ---------------- event navigation ----------------
    def GotoEvent(self, ev):
        if ev < 0 or ev >= self.fMaxEv:
            print(f"GotoEvent: invalid event id {ev}.")
            return False
        self.fCurEv = ev
        if self.fLoadTracks:
            self.LoadTracksFromTree()
        if self.fEnableQuickAnalysis:
            self.FillInvariantMassHistos()
            self.FillEnergyLossHisto()
        return True

    def NextEvent(self):
        if self.fCurEv < self.fMaxEv - 1:
            self.ResetMatrices()
            self.GotoEvent(self.fCurEv + 1)

    def PrevEvent(self):
        if self.fCurEv != 0:
            self.ResetMatrices()
            self.GotoEvent(self.fCurEv - 1)

    # ---------------- buttons of the "Analysis Tools" panel ----------------
    def LoadTracks(self):
        """'Load Tracks' check box: toggles fLoadTracks and reloads the current event."""
        self.fLoadTracks = not self.fLoadTracks
        self.GotoEvent(self.fCurEv)

    def SetPIDCutValues(self, p1, p2, de1, de2):
        """'Define Track Cuts' dialog: momentum range p1 < p < p2 (GeV/c) and energy loss de1 <= dE/dx < de2."""
        self.p1, self.p2, self.de1, self.de2 = F32(p1), F32(p2), F32(de1), F32(de2)

    def ApplyTrackCuts(self):
        """'Apply Track Cuts': from now on LoadTracksFromTree() drops tracks outside the cuts."""
        self.fApplyTrackCuts = True
        self.GotoEvent(self.fCurEv)

    def SwapApplyTrackCuts(self):
        if self.fApplyTrackCuts:
            self.fApplyTrackCuts = False
            self.GotoEvent(self.fCurEv)
        else:
            self.ApplyTrackCuts()

    def SwapQuickAnalysis(self):
        """'Quick Analysis' check box: when on, every event visited is filled automatically."""
        self.fEnableQuickAnalysis = not self.fEnableQuickAnalysis

    def EventsJump(self, n):
        """'Jump N Events' (only with Quick Analysis on): NextEvent() n times, if that stays inside the file."""
        if not self.fEnableQuickAnalysis:
            print('To use this function you have to enable "Quick Analysis".')
            return
        if n + self.fCurEv < self.fMaxEv:
            for _ in range(n):
                self.NextEvent()
        else:
            print("Too many events selected!")

    def ClearHisto(self):
        """'Clear Histograms'. As in the C++, nEvents is NOT reset, so a later Autosave() still counts the
        events filled before the clear."""
        for h in (self.gMinvHist, self.geeHist, self.gppHist, self.gJpsiHist):
            h.Reset()
        self.gEnergyLoss[:] = 0

    def BeginAgain(self):
        """'Reset': cuts back to 0, Load Tracks / Apply Track Cuts / Quick Analysis off, back to the first event.
        As in the C++, the histograms and nEvents are kept."""
        self.p1 = self.p2 = self.de1 = self.de2 = F32(0)
        self.fLoadTracks = self.fApplyTrackCuts = self.fEnableQuickAnalysis = False
        self.GotoEvent(0)

    # ---------------- track loading ----------------
    def LoadTracksFromTree(self):
        e = self.events[self.fCurEv]
        px, py, pz, dedx, charge = e['px'], e['py'], e['pz'], e['dedx'], e['charge']
        self.ResetMatrices()
        pSquared = px * px + py * py + pz * pz                                   # Float_t
        if self.fApplyTrackCuts:
            # if (dedx >= de2 || dedx < de1 || pSquared >= p2*p2 || pSquared <= p1*p1) -> drop the track
            drop = (dedx >= self.de2) | (dedx < self.de1) | (pSquared >= self.p2 * self.p2) | (pSquared <= self.p1 * self.p1)
            keep = ~drop
        else:
            keep = np.ones(len(dedx), dtype=bool)
        px, py, pz, dedx, charge, pSquared = px[keep], py[keep], pz[keep], dedx[keep], charge[keep], pSquared[keep]
        self.nSelectedTracks = int(keep.sum())
        self.gArrP = np.sqrt(pSquared.astype(np.float64)).astype(np.float32)   # TMath::Sqrt(px*px + py*py + pz*pz)
        self.gArrdEdx = dedx
        # if energy loss below threshold, assume particle is a pion, otherwise assume it is an electron
        massSquared = np.where(dedx > self.ELECTRON_DEDX, F32(0), self.PION_MASS_SQUARED).astype(np.float32)
        energy = np.sqrt((pSquared + massSquared).astype(np.float64)).astype(np.float32)
        four = np.stack([energy, px, py, pz], axis=1).astype(np.float32)
        self.po = four[charge > 0]          # if (charge > 0) -> positron list
        self.el = four[~(charge > 0)]       # else            -> electron list
        self.track_table = {'p': self.gArrP, 'dedx': dedx, 'charge': charge, 'E': energy, 'massSquared': massSquared}

    def ResetMatrices(self):
        self.el = np.zeros((0, 4), np.float32)
        self.po = np.zeros((0, 4), np.float32)

    # ---------------- histogram filling ----------------
    @staticmethod
    def pair_mass(a, b):
        """Float_t mass = TMath::Sqrt((E1+E2)^2 - (px1+px2)^2 - (py1+py2)^2 - (pz1+pz2)^2), evaluated left to right."""
        s0, s1, s2, s3 = a[:, 0] + b[:, 0], a[:, 1] + b[:, 1], a[:, 2] + b[:, 2], a[:, 3] + b[:, 3]
        arg = s0 * s0 - s1 * s1 - s2 * s2 - s3 * s3                              # Float_t
        with np.errstate(invalid='ignore'):
            return np.sqrt(arg.astype(np.float64)).astype(np.float32)             # negative argument -> NaN, as in C++

    def FillInvariantMassHistos(self):
        if not self.fLoadTracks:
            print("Load tracks before filling invariant mass histos!")
            return
        self.nEvents += 1
        el, po = self.el, self.po
        i, j = np.triu_indices(len(el), k=1)
        m_ee = self.pair_mass(el[i], el[j])
        i, j = np.triu_indices(len(po), k=1)
        m_pp = self.pair_mass(po[i], po[j])
        ie, jp = np.repeat(np.arange(len(el)), len(po)), np.tile(np.arange(len(po)), len(el))
        m_ep = self.pair_mass(el[ie], po[jp])
        self.geeHist.Fill(m_ee)
        self.gppHist.Fill(m_pp)
        self.gMinvHist.Fill(m_ep)
        # gJpsiHist gets this event's e-e+ pairs, then this event's e-e- and e+e+ pairs are subtracted
        ev = Hist1D(60, 0., 6.)
        ev.Fill(m_ep)
        ev_ee = Hist1D(60, 0., 6.); ev_ee.Fill(m_ee)
        ev_pp = Hist1D(60, 0., 6.); ev_pp.Fill(m_pp)
        self.gJpsiHist.Add(ev)
        self.gJpsiHist.Add(ev_ee, -1)
        self.gJpsiHist.Add(ev_pp, -1)
        self.last_event_pairs = {'el': el.copy(), 'po': po.copy(), 'm_ee': m_ee, 'm_pp': m_pp, 'm_ep': m_ep,
                                 'ep_index': (ie, jp)}
        if self.record_pairs:                                  # extension, not in masterclass.C
            pt = lambda a, b: np.hypot(a[:, 1].astype(float) + b[:, 1], a[:, 2].astype(float) + b[:, 2])
            self.pair_log['unlike'].append(np.stack([m_ep, pt(el[ie], po[jp])], axis=1))
            i, j = np.triu_indices(len(el), k=1)
            self.pair_log['like'].append(np.stack([m_ee, pt(el[i], el[j])], axis=1))
            i, j = np.triu_indices(len(po), k=1)
            self.pair_log['like'].append(np.stack([m_pp, pt(po[i], po[j])], axis=1))
        self.ResetMatrices()

    def FillEnergyLossHisto(self):
        if not self.fLoadTracks:
            print("Load tracks before filling energy loss histos!")
            return
        if self.nSelectedTracks > 0:
            ix = np.searchsorted(self.pid_xedges, self.gArrP, side='right') - 1
            iy = np.searchsorted(self.pid_yedges, self.gArrdEdx, side='right') - 1
            ok = (ix >= 0) & (ix < 200) & (iy >= 0) & (iy < 120)
            np.add.at(self.gEnergyLoss, (ix[ok], iy[ok]), 1.0)
            self.gArrP = np.zeros(0, np.float32)            # ResetDedxArray()
            self.gArrdEdx = np.zeros(0, np.float32)
            self.nSelectedTracks = 0

    # ---------------- 'Extract Signals' panel ----------------
    def CalculateIntegral(self, minm, maxm):
        """Integrate gJpsiHist (signal) and geeHist + gppHist (background) over the mass bins of the window."""
        self.minm, self.maxm = F32(minm), F32(maxm)
        massBin1 = int(self.gJpsiHist.FindBin(F32(self.minm + self.epsilon)))
        massBin2 = int(self.gJpsiHist.FindBin(F32(self.maxm - self.epsilon)))
        self.numJpsi = int(self.gJpsiHist.Integral(massBin1, massBin2))                          # Int_t numJpsi
        self.bg = int(self.geeHist.Integral(massBin1, massBin2) + self.gppHist.Integral(massBin1, massBin2))
        if self.numJpsi + self.bg > 0:
            self.significance = F32(self.numJpsi / np.sqrt(self.numJpsi + self.bg))
        # (as in the C++, the significance keeps its previous value if numJpsi + bg <= 0)
        if self.bg > 0:
            self.sb = F32(1. * self.numJpsi / self.bg)
        elif self.numJpsi > 0:
            self.sb = F32(9999.)
        return {'S': self.numJpsi, 'B': self.bg, 'S/B': float(self.sb), 'significance': float(self.significance),
                'bins': (massBin1, massBin2)}

    def Autosave(self, path=SAVE_FILE):
        """Append the current cuts, window, event count and results in masterclass.save's format; return the text."""
        g = lambda x: f"{float(x):.6g}"     # C++ ostream default: 6 significant digits
        text = (f"Momentum region: {g(self.p1)} < p < {g(self.p2)}\n"
                f"Specific energy loss: {g(self.de1)} < dE/dx < {g(self.de2)}\n"
                f"Invariant mass window: {g(self.minm)}< m <{g(self.maxm)}\n"
                f"Number of events = {self.nEvents}\n"
                f"Number of Jpsi's: {self.numJpsi}\n"
                f"Signal/Background: {g(self.sb)}\n"
                f"Significance: {g(self.significance)}\n\n")
        with open(path, 'a') as fh:
            fh.write(text)
        return text

    # ---------------- extension helpers (not in masterclass.C) ----------------
    def recorded_pairs(self):
        """Pairs kept with record_pairs=True: {'unlike'|'like': {'m': masses, 'pt': pair pT}}."""
        out = {}
        for kind, chunks in self.pair_log.items():
            a = np.concatenate(chunks) if chunks else np.zeros((0, 2))
            out[kind] = {'m': a[:, 0].astype(np.float32), 'pt': a[:, 1]}
        return out


def run_quick_analysis(dataset, cuts, n_events=None, record_pairs=False):
    """
    The clicks of a full masterclass run, in the order the manual gives them:
    Load Tracks -> Define Track Cuts -> Apply Track Cuts -> Fill Invariant Mass Histograms (event 1)
    -> Quick Analysis -> Jump N Events. cuts = (p1, p2, de1, de2), or None to analyse without track cuts.
    n_events = total number of events filled (default: every event in the file).
    """
    r = TVSDReader(dataset, record_pairs=record_pairs)
    n = r.fMaxEv if n_events is None else n_events
    r.LoadTracks()
    if cuts is not None:
        r.SetPIDCutValues(*cuts)
        r.ApplyTrackCuts()
    r.FillInvariantMassHistos()
    r.FillEnergyLossHisto()
    r.SwapQuickAnalysis()
    r.EventsJump(n - 1)
    assert r.nEvents == n
    return r

A drawing function shows the program's invariant-mass histograms the way ALICE plots them: zoomed on the J/$\psi$ region (1.5–4.5 GeV/$c^2$), opposite-charge pairs as points with statistical error bars on top of the same-charge background (shaded), and the background-subtracted signal with its error bars ($\sqrt{N_{e^-e^+} + N_{e^-e^-} + N_{e^+e^+}}$). The counts are the program's, in its 0.1 GeV/$c^2$ bins. Colours are used consistently in all J/$\psi$ plots: violet for opposite-charge pairs, grey for same-charge pairs, green for the signal, and in the comparison plots teal squares for pp and violet circles for p–Pb.

In [ ]:
COL_UNLIKE, COL_LIKE, COL_SIGNAL, COL_PP, COL_PPB = '#7B5CD6', '#5B6274', '#3E9B4F', '#1EA7A0', '#7B5CD6'

MASS_VIEW = (1.5, 4.5)   # GeV/c^2: the J/psi region shown in the plots (the histograms themselves run 0-6)
SHOW_FULL_MASS_RANGE = False   # toggle: True shows every mass plot over the full 0-6 GeV/c^2 range

def DrawInvMass(r, title, window=None, full_range=None):
    """
    ALICE-style invariant-mass plots, zoomed on the J/psi region, from the program's histograms:
      left : opposite-charge pairs N(e-e+) as points with statistical errors sqrt(N), on top of the
             same-charge background N(e-e-) + N(e+e+) as a filled histogram
      right: the background-subtracted signal N(e-e+) - N(e-e-) - N(e+e+) as points with errors
             sqrt(N(e-e+) + N(e-e-) + N(e+e+))
    The counting is unchanged: the program's 0.1 GeV/c^2 bins.
    full_range: True shows 0-6 GeV/c^2, False the J/psi region; None follows SHOW_FULL_MASS_RANGE.
    """
    if full_range is None:
        full_range = SHOW_FULL_MASS_RANGE
    mview = (0.0, 6.0) if full_range else MASS_VIEW
    edges = r.gMinvHist.edges()
    centers = 0.5 * (edges[:-1] + edges[1:])
    N = r.gMinvHist.contents()
    B = r.geeHist.contents() + r.gppHist.contents()
    S = N - B
    view = (centers > mview[0]) & (centers < mview[1])
    k0, k1 = np.where(view)[0][[0, -1]]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
    ax1.stairs(B[k0:k1 + 1], edges[k0:k1 + 2], fill=True, color=COL_LIKE, alpha=0.35,
               label='e$^-$e$^-$ + e$^+$e$^+$ (same charge)')
    ax1.stairs(B[k0:k1 + 1], edges[k0:k1 + 2], color=COL_LIKE, linewidth=1.2)
    ax1.errorbar(centers[view], N[view], yerr=np.sqrt(N[view]), fmt='o', color=COL_UNLIKE, markersize=4.5,
                 capsize=2.5, label='e$^-$e$^+$ (opposite charge)')
    ax1.set_ylim(0, 1.35 * max(5, (N[view] + np.sqrt(N[view])).max(), B[view].max()))
    ax1.set_ylabel(r'pairs per 0.1 $\mathrm{GeV}/c^2$')
    ax2.errorbar(centers[view], S[view], yerr=np.sqrt(N[view] + B[view]), fmt='o', color=COL_SIGNAL, markersize=4.5,
                 capsize=2.5, label='signal: e$^-$e$^+$ $-$ (e$^-$e$^-$ + e$^+$e$^+$)')
    ax2.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    top = (S[view] + np.sqrt(N[view] + B[view])).max()
    ax2.set_ylim(min(0, (S[view] - np.sqrt(N[view] + B[view])).min()) * 1.15, 1.35 * max(5, top))
    ax2.set_ylabel(r'signal pairs per 0.1 $\mathrm{GeV}/c^2$')
    for ax in (ax1, ax2):
        if window is not None:
            ax.axvspan(*window, color=COL_SIGNAL, alpha=0.12, label='mass window')
        ax.set_xlim(*mview)
        ax.set_xlabel(r'$m_{ee}$ ($\mathrm{GeV}/c^2$)', fontsize=14)
        label = title if ax is ax1 else 'after background subtraction'
        ax.text(0.03, 0.96, f"ALICE J/$\\psi$ masterclass data\n{label}\n{r.nEvents:,} events",
                transform=ax.transAxes, va='top', fontsize=9)
        ax.legend(loc='upper right', fontsize=8.5)
    plt.tight_layout()
    plt.show()


### C.5 — Validation: reproducing every result the program saved

Each time a student presses "Extract Signals", the program appends the cuts, mass window, event count and results to `masterclass.save`. The copy in the zip holds nine such results from earlier masterclass runs. The log does not say which events were filled, but searching all event ranges for the saved numbers singles out three sessions. The cell below replays them click by click with the port and checks that every saved block is reproduced **character for character** (the `assert` fails otherwise).

The replay also explains the odd event counts in the log. "Clear Histograms" empties the histograms but leaves `nEvents` running, so, for example, "Number of events = 7001" is the 3,501 events of the previous run plus the 3,500 events actually in the histograms.

In [ ]:
# Validation: replay the sessions logged in masterclass.save (shipped in the zip) and compare every saved block.
# The log does not record which events were filled. Searching event ranges for the saved numbers shows they come
# from three sessions, replayed below; ClearHisto() empties the histograms but does not reset nEvents, which is
# why some logged event counts are larger than the number of events actually in the histograms.
saved_blocks = open(os.path.join(MASTERCLASS_DIR, 'masterclass.save')).read().strip().split('\n\n')
saved_blocks = [b.strip() + '\n' for b in saved_blocks]
replayed = []

def autosave_text(r):
    return r.Autosave(os.devnull).strip() + '\n'

# Session 1 (pp): 204 fills that were then cleared (the count follows from the log: 1204 - 1000); cuts 1 < p < 10,
# 65 <= dE/dx < 80; from event 1 jump 1000 events (events 2-1001 as the program numbers them); five mass windows saved.
r = TVSDReader(1)
r.LoadTracks()
r.SetPIDCutValues(1, 10, 65, 80); r.ApplyTrackCuts()
r.nEvents = 204                              # fills made before ClearHisto(): counted in nEvents, not in the histograms
r.SwapQuickAnalysis(); r.EventsJump(1000)
for window in [(2, 4), (2, 4), (1, 4), (1, 5), (2.9, 3.2)]:
    r.CalculateIntegral(*window); replayed.append(autosave_text(r))
# Session 2 (pp), a fresh start: cuts 1 < p < 10, 65 <= dE/dx < 90; event 1 filled, then jump 3500 events (events 1-3501)
r = TVSDReader(1)
r.LoadTracks()
r.SetPIDCutValues(1, 10, 65, 90); r.ApplyTrackCuts()
r.FillInvariantMassHistos(); r.SwapQuickAnalysis(); r.EventsJump(3500)
r.CalculateIntegral(2.8, 3.2); replayed.append(autosave_text(r))
# ... then Clear + Reset (cuts back to 0, no track cuts): from event 1 jump 3500 events (events 2-3501)
r.ClearHisto(); r.BeginAgain(); r.LoadTracks(); r.SwapQuickAnalysis(); r.EventsJump(3500)
r.CalculateIntegral(2.8, 3.2); replayed.append(autosave_text(r))

# Session 3 (p-Pb), no track cuts: event 1 filled, histograms cleared, then jump 2000 events (events 2-2001)
r = TVSDReader(2)
r.LoadTracks(); r.FillInvariantMassHistos(); r.ClearHisto()
r.SwapQuickAnalysis(); r.EventsJump(2000)
r.CalculateIntegral(2.8, 3.2); replayed.append(autosave_text(r))
# ... then Clear + Reset, cuts 1 < p < 10, 65 <= dE/dx < 90, event 1 filled, jump 2000 events
r.ClearHisto(); r.BeginAgain(); r.LoadTracks()
r.SetPIDCutValues(1, 10, 65, 90); r.ApplyTrackCuts()
r.FillInvariantMassHistos(); r.SwapQuickAnalysis(); r.EventsJump(2000)
r.CalculateIntegral(2.8, 3.2); replayed.append(autosave_text(r))

assert len(replayed) == len(saved_blocks) == 9
for k, (mine, theirs) in enumerate(zip(replayed, saved_blocks), 1):
    assert mine == theirs, (k, mine, theirs)
print("All 9 results in masterclass.save reproduced character for character, e.g.:\n")
print(replayed[5])

### C.6 — Manual step 1: load some events and look at their tracks

The manual starts with: choose "p-p collision" in the start menu, then **Load Tracks** and step through a few events. Each track has a momentum $\vec p = (p_x, p_y, p_z)$, an energy loss dE/dx and a charge. `LoadTracksFromTree()` also gives every track an assumed mass (C.1, step 3) and energy $E = \sqrt{p^2 + m^2}$.

In [ ]:
reader = TVSDReader(1)      # start menu: "p-p collision"
reader.LoadTracks()         # 'Load Tracks': the tracks of event 1 are read

def track_dataframe(r):
    t = r.track_table
    return pd.DataFrame({'p (GeV/c)': t['p'].round(3), 'dE/dx': t['dedx'].astype(int), 'charge': t['charge'],
                         'treated as': np.where(t['dedx'] > 62, 'electron', 'pion'),
                         'm^2 (GeV^2)': t['massSquared'].round(3), 'E (GeV)': t['E'].round(3)})

print(f"Event {reader.fCurEv + 1} of {reader.fMaxEv}: {len(reader.track_table['p'])} tracks")
display(track_dataframe(reader))

for _ in range(4):          # the 'next event' arrow, four times
    reader.NextEvent()
    t = reader.track_table
    print(f"Event {reader.fCurEv + 1}: {len(t['p']):3d} tracks, {int(np.sum(t['dedx'] > 62)):2d} with dE/dx > 62 "
          f"({int(np.sum(t['charge'] < 0))} negative, {int(np.sum(t['charge'] > 0))} positive tracks in total)")

The table shows why the program's mass rule alone is not a particle identification: it treats every track with dE/dx $> 62$ as an electron, including slow protons and kaons, which lose much more energy (a track with dE/dx near 300 at $p \approx 0.4$ GeV/$c$ above). Step 2 therefore uses an upper dE/dx cut as well, and step 3 a minimum momentum.

### C.7 — Manual step 2: where are the electrons?

Step 2 asks for track cuts that keep electrons and positrons and reject pions, kaons and protons. The quantity that separates them is the energy loss dE/dx (C.1, step 5). First, the reason cuts are needed at all: a Quick Analysis over the whole pp file with no track cuts. Without cuts, almost every pair is two pions and the J/$\psi$ cannot be seen.

In [ ]:
reader_nocut = run_quick_analysis(1, cuts=None)     # Quick Analysis over all pp events, no track cuts
res = reader_nocut.CalculateIntegral(2.8, 3.2)
print(f"No track cuts, 2.8 < m < 3.2 GeV/c^2: S = {res['S']}, B = {res['B']}, S/B = {res['S/B']:.4f}, "
      f"significance = {res['significance']:.2f}")
DrawInvMass(reader_nocut, 'pp, no track cuts: the J/psi is lost in the background', window=(2.8, 3.2))

Where the electrons sit in dE/dx can be read off the tracks directly. At momenta where electrons and pions are well separated, $2 < p < 5$ GeV/$c$, the table counts the tracks in a few dE/dx ranges, gives the most common dE/dx of the pions (below 62, the program's electron threshold), and the median dE/dx of the electron band (tracks with $68 \le$ dE/dx $< 100$, above the pions' tail). Pions are the large group near 55; electrons are the smaller group near 75, which the cuts $65 \le$ dE/dx $< 90$ enclose.

In [ ]:
def dedx_summary(dataset):
    evs = read_vsd_events(dataset)
    p = np.concatenate([np.sqrt(e['px'].astype(float)**2 + e['py']**2 + e['pz']**2) for e in evs])
    d = np.concatenate([e['dedx'] for e in evs])[(p > 2) & (p < 5)]
    row = {'tracks, dE/dx < 62': int(np.sum(d < 62)), '62-65': int(np.sum((d >= 62) & (d < 65))),
           '65-90': int(np.sum((d >= 65) & (d < 90))), '90 and up': int(np.sum(d >= 90))}
    row['pion peak: most common dE/dx below 62'] = int(np.bincount(d[d < 62].astype(int)).argmax())
    row['electron band: median dE/dx of 68-100'] = float(np.median(d[(d >= 68) & (d < 100)]))
    return row

display(pd.DataFrame([dedx_summary(1)], index=['pp, tracks with 2 < p < 5 GeV/c']))

### C.8 — Manual step 3: add a momentum cut, then define and apply the cuts

C.1, step 4 showed that J/$\psi$ electrons have momenta around $m_{J/\psi}/2 \approx 1.55$ GeV/$c$. Electrons from other sources, such as photon conversions, are mostly softer, so a minimum momentum removes many of them. The cuts used here are the ones the saved masterclass runs used:

$$1 < p < 10 \text{ GeV}/c, \qquad 65 \le \text{dE/dx} < 90 .$$

"Define Track Cuts" sets them and "Apply Track Cuts" switches them on; from then on only tracks inside the cuts are loaded. C.10 checks other choices.

In [ ]:
JPSI_CUTS = {1: (1.0, 10.0, 65, 90)}     # (p1, p2, de1, de2) per dataset; p-Pb is added in C.12

reader = TVSDReader(1)
reader.LoadTracks()
reader.SetPIDCutValues(*JPSI_CUTS[1])    # 'Define Track Cuts'
reader.ApplyTrackCuts()                  # 'Apply Track Cuts'

print(f"Event {reader.fCurEv + 1}: {len(reader.track_table['p'])} of {len(read_vsd_events(1)[0]['dedx'])} tracks pass the cuts")
display(track_dataframe(reader))
n_all = sum(len(e['dedx']) for e in read_vsd_events(1))
n_kept = 0
for ev in range(reader.fMaxEv):                 # count over the whole file, event by event
    reader.GotoEvent(ev)
    n_kept += len(reader.track_table['p'])
print(f"All pp events: {n_kept:,} of {n_all:,} tracks pass the cuts")

### C.9 — Manual step 4: invariant masses and the J/$\psi$ signal

With the cuts applied, **Fill Invariant Mass Histograms** pairs up the selected tracks of the current event. First, one pair calculated by hand: the cell finds the first event with an opposite-charge pair inside $2.8 < M < 3.2$ GeV/$c^2$ and works through C.1, steps 1–3 for its two tracks, then compares with the port's value.

In [ ]:
# Find the first event (with the cuts applied) that has an e-e+ pair in 2.8 < m < 3.2 GeV/c^2
probe = TVSDReader(1)
probe.LoadTracks(); probe.SetPIDCutValues(*JPSI_CUTS[1]); probe.ApplyTrackCuts()
while True:
    probe.FillInvariantMassHistos()
    lp = probe.last_event_pairs
    hit = np.where((lp['m_ep'] >= 2.8) & (lp['m_ep'] < 3.2))[0]
    if len(hit):
        break
    probe.NextEvent()

k = hit[0]
ie, jp = lp['ep_index']
e1, e2 = lp['el'][ie[k]].astype(float), lp['po'][jp[k]].astype(float)
print(f"Event {probe.fCurEv + 1}: electron candidate (negative) and positron candidate (positive)\n")
for name, v in [('e-', e1), ('e+', e2)]:
    p = np.sqrt(v[1]**2 + v[2]**2 + v[3]**2)
    print(f"  {name}: p = ({v[1]:+.4f}, {v[2]:+.4f}, {v[3]:+.4f}) GeV/c,  |p| = {p:.4f} GeV/c,  "
          f"E = sqrt(|p|^2 + 0) = {v[0]:.4f} GeV")
E_sum = e1[0] + e2[0]
P_sum = e1[1:] + e2[1:]
M2 = E_sum**2 - np.sum(P_sum**2)
print(f"\n  E1 + E2       = {e1[0]:.4f} + {e2[0]:.4f} = {E_sum:.4f} GeV")
print(f"  p1 + p2       = ({P_sum[0]:+.4f}, {P_sum[1]:+.4f}, {P_sum[2]:+.4f}) GeV/c,  |p1 + p2|^2 = {np.sum(P_sum**2):.4f} GeV^2")
print(f"  M^2           = {E_sum:.4f}^2 - {np.sum(P_sum**2):.4f} = {M2:.4f} GeV^2")
print(f"  M             = sqrt({M2:.4f}) = {np.sqrt(M2):.4f} GeV/c^2")
print(f"  port's value  = {lp['m_ep'][k]:.4f} GeV/c^2   (J/psi mass: 3.097 GeV/c^2)")
cos12 = np.dot(e1[1:], e2[1:]) / (np.linalg.norm(e1[1:]) * np.linalg.norm(e2[1:]))
print(f"  check, massless formula 2 p1 p2 (1 - cos theta): "
      f"{np.sqrt(2 * np.linalg.norm(e1[1:]) * np.linalg.norm(e2[1:]) * (1 - cos12)):.4f} GeV/c^2 "
      f"(opening angle {np.degrees(np.arccos(cos12)):.1f} degrees)")

One pair in the window is not yet a discovery: it could be a random combination. The manual's next action is to fill a number of events one at a time and press **Extract Signals** for a mass window around the J/$\psi$ mass. Here the first 100 events are filled the way a student would, with the arrow and the fill button, and the result is saved with `Autosave()` (to `masterclass_notebook.save`, so the zip's `masterclass.save` is left as it is).

In [ ]:
JPSI_WINDOW = (2.8, 3.2)    # GeV/c^2

reader = TVSDReader(1)
reader.LoadTracks(); reader.SetPIDCutValues(*JPSI_CUTS[1]); reader.ApplyTrackCuts()
for ev in range(100):
    reader.FillInvariantMassHistos()     # 'Fill Invariant Mass Histograms'
    reader.FillEnergyLossHisto()         # 'Fill PID Histograms'
    reader.NextEvent()                   # next-event arrow

res = reader.CalculateIntegral(*JPSI_WINDOW)     # 'Extract Signals'
print(f"After {reader.nEvents} events, window {JPSI_WINDOW[0]} < m < {JPSI_WINDOW[1]} GeV/c^2 (bins {res['bins'][0]}-{res['bins'][1]}):")
print(f"  S = N(e-e+) - B = {res['S'] + res['B']} - {res['B']} = {res['S']} J/psi")
print(f"  S/B = {res['S']}/{res['B']} = {res['S/B']:.4g},   significance = {res['S']}/sqrt({res['S'] + res['B']}) = {res['significance']:.3f}\n")
if os.path.exists(SAVE_FILE):
    os.remove(SAVE_FILE)                 # start a fresh log for this notebook
print(reader.Autosave())
DrawInvMass(reader, 'pp, first 100 events, cuts applied', window=JPSI_WINDOW)

### C.10 — Manual step 5: refine the cuts and the mass window

The manual's step 5: if the significance is poor or the background too high, change the track cuts or the mass window and try again. The tables below do this systematically on the first 1,000 pp events (Quick Analysis for each setting). Each row is one complete run with the given cuts, the same mass window $2.8 < M < 3.2$ GeV/$c^2$.

A larger window catches more of the peak (larger $S$) but also more background (larger $B$); tighter cuts lower $B$ but lose electrons. The significance balances the two. One caution: picking the best of many settings on the same events favours settings where the background happened to fluctuate down, so the best significance in a scan is slightly optimistic.

In [ ]:
N_SCAN = 1000
rows = []
for p1 in [0.5, 1.0, 1.5]:
    for de1 in [60, 65, 70]:
        for de2 in [80, 90, 100]:
            r = run_quick_analysis(1, (p1, 10.0, de1, de2), n_events=N_SCAN)
            res = r.CalculateIntegral(*JPSI_WINDOW)
            rows.append({'p > (GeV/c)': p1, 'dE/dx from': de1, 'dE/dx below': de2, 'S': res['S'], 'B': res['B'],
                         'S/B': round(res['S/B'], 3), 'significance': round(res['significance'], 2)})
scan = pd.DataFrame(rows).sort_values('significance', ascending=False).reset_index(drop=True)
print(f"Track-cut scan, first {N_SCAN} pp events, window {JPSI_WINDOW[0]} < m < {JPSI_WINDOW[1]} GeV/c^2 (best first):")
display(scan.head(10))
print("Chosen cuts 1 < p < 10, 65 <= dE/dx < 90:")
display(scan[(scan['p > (GeV/c)'] == 1.0) & (scan['dE/dx from'] == 65) & (scan['dE/dx below'] == 90)])

r = run_quick_analysis(1, JPSI_CUTS[1], n_events=N_SCAN)
wrows = []
for lo, hi in [(2.0, 4.0), (2.5, 3.5), (2.7, 3.3), (2.8, 3.2), (2.9, 3.2), (3.0, 3.2)]:
    res = r.CalculateIntegral(lo, hi)
    wrows.append({'window (GeV/c^2)': f'{lo} - {hi}', 'S': res['S'], 'B': res['B'], 'S/B': round(res['S/B'], 3),
                  'significance': round(res['significance'], 2)})
print(f"Mass-window scan, chosen cuts, first {N_SCAN} pp events:")
display(pd.DataFrame(wrows))

The cuts $1 < p < 10$ GeV/$c$, $65 \le$ dE/dx $< 90$ are among the best settings in the scan; the few settings close to them differ by much less than the statistical precision of 1,000 events. In the window scan, $2.7 < M < 3.3$ GeV/$c^2$ comes out marginally ahead of $2.8 < M < 3.2$ GeV/$c^2$, again by much less than the statistical precision. This notebook keeps $2.8 < M < 3.2$ GeV/$c^2$, the window of the saved masterclass runs, together with these cuts.

### C.11 — Manual step 6: Quick Analysis over all events

With cuts that work on a sample, the manual's last step is **Quick Analysis**: switch it on and use **Jump N Events** to fill many events at once. Here that is every event in the file: event 1 is filled by hand, then a jump of 3,864 events fills the rest.

In [ ]:
jpsi_reader = {}
jpsi_reader['pp'] = run_quick_analysis(1, JPSI_CUTS[1])
res = jpsi_reader['pp'].CalculateIntegral(*JPSI_WINDOW)
print(f"pp, all {jpsi_reader['pp'].nEvents:,} events: S = {res['S']} J/psi, B = {res['B']}, "
      f"S/B = {res['S/B']:.3f}, significance = {res['significance']:.2f}\n")
print(jpsi_reader['pp'].Autosave())
DrawInvMass(jpsi_reader['pp'], 'pp, all events, cuts 1 < p < 10 GeV/c, 65 ≤ dE/dx < 90', window=JPSI_WINDOW)

The opposite-charge pairs (left, points) have a clear peak at the J/$\psi$ mass that the same-charge background (shaded) does not have. After subtraction (right), the peak stands alone. With a significance of about 14, this is far beyond the "5 sigma" that particle physicists require for a discovery.

#### What the cuts do to the shape of the mass spectrum

The plots above are zoomed on the J/$\psi$ region. Over the full range the program fills, 0–6 GeV/$c^2$, a dielectron mass spectrum is expected to have its largest peak at zero mass and to fall steadily, with a small J/$\psi$ peak on top. The pp spectrum with the cuts used here does not look like that. The cell below shows why: it runs the same events three times — with no track cuts, with the dE/dx cut only, and with the full cuts (dE/dx and momentum) — and draws the full range. Set `SHOW_FULL_MASS_RANGE = True` (in C.4) to draw every mass plot in this notebook over the full range.

In [ ]:
def mass_spectra(r):
    N = r.gMinvHist.contents()
    B = r.geeHist.contents() + r.gppHist.contents()
    return N, B

p1, p2, de1, de2 = JPSI_CUTS[1]
reader_dedx_only = run_quick_analysis(1, (0.0, 1.0e6, de1, de2))   # dE/dx cut only: no momentum cut
selections = [(reader_nocut, 'no track cuts'),
              (reader_dedx_only, f'dE/dx cut only: {de1:g} ≤ dE/dx < {de2:g}'),
              (jpsi_reader['pp'], f'full cuts: {p1:g} < p < {p2:g} GeV/c, {de1:g} ≤ dE/dx < {de2:g}')]

edges = reader_nocut.gMinvHist.edges()
centers = 0.5 * (edges[:-1] + edges[1:])
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharex=True)
for col, (r, label) in enumerate(selections):
    N, B = mass_spectra(r)
    S, err = N - B, np.sqrt(N + B)
    ax = axes[0, col]
    ax.stairs(B, edges, fill=True, color=COL_LIKE, alpha=0.35, label='e$^-$e$^-$ + e$^+$e$^+$ (same charge)')
    ax.stairs(B, edges, color=COL_LIKE, linewidth=1.0)
    ax.errorbar(centers, N, yerr=np.sqrt(N), fmt='o', color=COL_UNLIKE, markersize=3.5, capsize=2,
                label='e$^-$e$^+$ (opposite charge)')
    ax.set_ylim(0, 1.3 * (N + np.sqrt(N)).max())
    ax.set_ylabel(r'pairs per 0.1 $\mathrm{GeV}/c^2$')
    ax.text(0.03, 0.96, f"pp, {r.nEvents:,} events\n{label}", transform=ax.transAxes, va='top', fontsize=9)
    ax.legend(loc='center right', fontsize=8)
    ax = axes[1, col]
    ax.errorbar(centers, S, yerr=err, fmt='o', color=COL_SIGNAL, markersize=3.5, capsize=2)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.axvspan(*JPSI_WINDOW, color=COL_SIGNAL, alpha=0.12)
    ax.set_ylabel(r'signal pairs per 0.1 $\mathrm{GeV}/c^2$')
    ax.set_xlabel(r'$m_{ee}$ ($\mathrm{GeV}/c^2$)', fontsize=14)
    ax.set_xlim(0, 6)
    ax.text(0.97, 0.96, 'after background subtraction', transform=ax.transAxes, va='top', ha='right', fontsize=9)
fig.suptitle('Invariant mass over the full range: the effect of the track cuts (pp)')
plt.tight_layout()
plt.show()

first = ['0.0-0.1', '0.1-0.2', '0.5-0.6', '1.0-1.1', '3.0-3.1']
idx = [0, 1, 5, 10, 30]
display(pd.DataFrame({label: [int(mass_spectra(r)[0][k]) for k in idx] for r, label in selections},
                     index=[f'{m} GeV/c²' for m in first]).rename_axis('opposite-charge pairs in bin'))

- **No track cuts:** almost every track is a pion and is given the pion mass (only tracks with dE/dx $> 62$ are treated as electrons), so a pion pair can never be lighter than $2m_\pi \approx 0.28$ GeV/$c^2$. The spectrum rises to a broad hump at 0.5–0.7 GeV/$c^2$ and then falls steadily. The J/$\psi$ is in there, but under tens of thousands of random pion pairs per bin.
- **dE/dx cut only:** this keeps the electrons and gives the expected shape: the largest peak in the first bin, 0–0.1 GeV/$c^2$, a shoulder up to about 1 GeV/$c^2$, then a steady fall with the J/$\psi$ peak at 3.0–3.1 GeV/$c^2$. The zero-mass peak is mostly photon conversions in the detector material ($\gamma \to e^-e^+$) and $\pi^0$ Dalitz decays ($\pi^0 \to \gamma e^-e^+$): the electron and positron are produced almost parallel, so their pair mass is close to zero. These are genuine opposite-charge pairs, so the peak survives the like-sign subtraction.
- **Full cuts:** the lower momentum cut, $p > 1$ GeV/$c$, removes the soft electrons, and with them most low-mass pairs: two tracks with $p > 1$ GeV/$c$ only form a light pair if they are nearly parallel. Below about 1 GeV/$c^2$ the spectrum is almost empty, apart from a few conversion pairs in the first bin. The J/$\psi$ peak keeps nearly all its pairs (J/$\psi$ electrons carry about 1.5 GeV/$c$ each), while the background under it drops. The momentum cut is chosen to improve the J/$\psi$ signal, not to show the low-mass region, so the shape of the spectrum with full cuts is expected and is not an error.

### C.12 — The same analysis on p–Pb collisions (start-menu choice 2)

The program's second dataset holds proton–lead collisions. Running the same six steps shows one new feature: **the electron band does not sit at exactly the same dE/dx** as in the pp data. The two datasets were recorded at different times with different detector calibrations: in the table below the median of the p–Pb electron band is a few units higher than in pp. The pp cuts therefore cut into the p–Pb electron band, and moving the lower dE/dx cut up to 70 follows it.

In [ ]:
display(pd.DataFrame([dedx_summary(1), dedx_summary(2)],
                     index=['pp, tracks with 2 < p < 5 GeV/c', 'p–Pb, tracks with 2 < p < 5 GeV/c']))

for cuts, label in [((1.0, 10.0, 65, 90), 'pp cuts, 65 <= dE/dx < 90'), ((1.0, 10.0, 70, 90), 'moved up, 70 <= dE/dx < 90')]:
    r = run_quick_analysis(2, cuts)
    res = r.CalculateIntegral(*JPSI_WINDOW)
    print(f"p-Pb, {label}: S = {res['S']}, B = {res['B']}, S/B = {res['S/B']:.3f}, significance = {res['significance']:.2f}")

JPSI_CUTS[2] = (1.0, 10.0, 70, 90)
jpsi_reader['pPb'] = run_quick_analysis(2, JPSI_CUTS[2])
jpsi_reader['pPb'].CalculateIntegral(*JPSI_WINDOW)
print('\n' + jpsi_reader['pPb'].Autosave())
DrawInvMass(jpsi_reader['pPb'], 'p–Pb, all events, cuts 1 < p < 10 GeV/c, 70 ≤ dE/dx < 90', window=JPSI_WINDOW)
print("Everything saved by this notebook (masterclass_notebook.save):\n")
print(open(SAVE_FILE).read())

---
## Download the saved plots

All plots from this run are in `figures/`. The cell below lists them and packs the folder into `figures.zip`, one file to download.

In [ ]:
import shutil

saved = sorted(f for f in os.listdir(FIG_DIR) if f.endswith('.pdf'))
print(f'{len(saved)} plots in {FIG_DIR}/:')
for f in saved:
    print('  ', f[:-4])
archive = shutil.make_archive(FIG_DIR, 'zip', FIG_DIR)
print(f'\nAll plots packed into {archive}')


---
## Summary

- **Part A**: the original charged-hadron $R_{AA}$/$R_{CP}$ pipeline, still runs on the original data files by default, with three corrections: the statistical uncertainty of the spectra includes the bin width, the $p_T$ bin edges are the true edges of the pp reference, and the pp uncertainties are the pp histogram's own errors; Section 3.5 compares $R_{AA}$ with ALICE's published values.
- **Part B**: the same maths, in one reusable, species-agnostic function — verified numerically identical to Part A.
- **Part C**: the ALICE J/$\psi$ masterclass — six steps on real pp (7 TeV) and p–Pb collisions: tracks, electron selection by energy loss and momentum, invariant masses, like-sign background subtraction, signal, S/B and significance, and the Quick Analysis over every event.
